# Shared neural dynamics between two interacting RNN agents

>**Zhang et al., "[Inter-brain neural dynamics in biological and artificial intelligence systems](https://doi.org/10.1038/s41586-025-09196-4)", *Nature* 2025**

In this notebook, two recurrent neural network agents — a *chaser* and an *explorer* — are trained with multi-agent reinforcement learning (MARL) in a shared grid-world arena, and we then ask the question: **do two interacting agents develop a shared neural subspace, and is that subspace causally relevant for social behavior?**

Everything here is built on the NeuralRNN framework: the agents are framework `ActorCriticModel`s wrapped around a vanilla continuous-time RNN core (`CTRNN` with $\alpha=1$, i.e. a discrete-time ReLU RNN), checkpoints are saved and loaded with `save_pretrained` / `AutoModel.from_pretrained`, and the returns machinery (`GAE`) comes from `neuralrnn.rl`. Only three pieces are specific to this study and therefore live in the notebook: the chase arena, a two-agent PPO trainer (the framework `RLTrainer` is single-agent), and the inter-agent analyses (PLSC, decoding, perturbation).

**Roadmap.** We first define the task and look at an example episode; then build the two agents and the training algorithm; then look at the learned *behavior* (training curves, an animation of the trained pair at work, movement flow fields — the analogs of Fig. 5 of the paper); and finally analyze the *neural* dynamics (event-aligned activity, SVM decoding, shared/unique subspaces via PLSC, partner representation, and the causal perturbation experiment — the analogs of Fig. 6).

## Setup

In [ ]:
import json
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch
from torch.distributions import Categorical

import neuralrnn  # registers all model families (import side effect)
from neuralrnn import AutoModel
from neuralrnn.rl.returns import GAE

# ----- paths (load-first protocol) -----------------------------------------
MODEL_DIR = Path("models/environment_chase")
FIG_DIR = Path("figs/environment_chase")
ANALYSIS_DIR = MODEL_DIR / "analysis"       # cached rollouts / analysis results
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

PAIR_DIRS = {
    "social": MODEL_DIR / "social" / "pair_seed0",
    "nonsocial": MODEL_DIR / "nonsocial" / "pair_seed0",
}

# ----- plotting style -------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False,
})
# event colors (paper palette)
EVENT_COLORS = {"none": "white", "new_field": "#4ecdc4", "collision": "#e74c3c",
                "approach": "#f5a623", "escape_far": "#e377c2",
                "escape_near": "#f2b6de", "escape_close": "#d98cb3"}
COLOR_SOCIAL, COLOR_NONSOCIAL = "#d62728", "#1f77b4"

def savefig(fig, name):
    fig.savefig(FIG_DIR / name)
    print("saved:", FIG_DIR / name)

print(f"device: {DEVICE}")

## The chase task

Two agents move on a $10\times10$ grid. The **explorer** ("agent 1") is rewarded for visiting tiles it has not visited before; the **chaser** ("agent 2") is rewarded for catching the explorer. An episode lasts 100 time steps; positions are re-drawn at random (and non-overlapping) at the start of every episode.

**Actions and movement.** Each agent chooses one of four actions per step — up, right, down, left. The chaser moves first. A move onto a wall is clamped (the agent stays); a move onto the partner's *current* tile is blocked — this is a **collision** — so the two agents never occupy the same tile.

**Observations.** Each agent sees a 200-dimensional vector: a 100-dimensional one-hot of its own position concatenated with a 100-dimensional one-hot of the partner's position. Crucially, the partner one-hot is gated by a $7\times7$ **field of vision** (FOV) centered on the agent (Chebyshev distance $\le 3$): when the partner is outside the FOV, the partner half of the observation is all zeros. The agents must therefore *search* for each other, and the chaser can only chase what it sees.

**Rewards** (per step):

| event | explorer | chaser |
|---|---|---|
| collision (chaser reaches explorer) | $-1$ | $+1$ |
| new field (first visit to a tile) | $+1$ | $+0.1$ |
| otherwise | $-0.5$ | $-0.1$ |

In the **non-social** control condition the collision term is removed from both reward functions and the partner is invisible (the partner half of the observation is always zero), so each agent effectively explores alone. Two further control conditions keep the non-social rewards but give the agents partial ($7\times7$ FOV) or full vision of the partner; we use them later to ask whether vision alone, without social reward pressure, produces shared dynamics.

**Events.** For the analyses we classify every time step (in priority order) as *collision*, *new field*, or a distance-based social event relative to the partner's previous position: the explorer *escapes* when it increases its distance to the chaser (thresholds at 5/3 tiles split far/near/close escapes), and the chaser *approaches* when it decreases its distance to the explorer.

The arena below is a faithful re-implementation of the reference environment (all variants parameterized by `social`, `partner_vision`, and `random_agent`), differentially tested step-for-step against the original code.

In [ ]:
import numpy as np

ACTIONS = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}  # up/right/down/left
OBS_DIM = 200
N_ACTIONS = 4

# Event column order used when events are exported as one-hot rows (mirrors the
# Bv1/Bv2 matrices of the reference evaluation script).
EVENT_COLUMNS = ["none", "new_field", "collision", "approach",
                 "escape_far", "escape_near", "escape_close"]


class ChaseArena:
    """Two-agent chase arena.

    Args:
        width, height: grid dimensions (paper: 10 x 10).
        episode_len: fixed episode horizon in steps (100 for training,
            500 for evaluation).
        social: True -> social reward structure (collision: explorer -1,
            chaser +1); False -> non-social (collision leaves both at the
            baseline penalty).
        partner_vision: "fov7" (partner one-hot only within the 7x7 field of
            view), "none" (partner input removed), or "full" (always present).
        random_agent: None, 1, or 2 — overwrite that agent's action with a
            uniform random draw (random-opponent playoff arenas).
        seed: seed for the arena RNG (position resets and random-opponent
            draws). Uses the global numpy RNG when None.
    """

    def __init__(self, width: int = 10, height: int = 10,
                 episode_len: int = 100, social: bool = True,
                 partner_vision: str = "fov7", random_agent: int | None = None,
                 seed: int | None = None):
        assert partner_vision in ("fov7", "none", "full")
        assert random_agent in (None, 1, 2)
        self.width = width
        self.height = height
        self.timestep_limit = episode_len
        self.social = social
        self.partner_vision = partner_vision
        self.random_agent = random_agent
        self.vision_side = 3  # (vision_field - 1) / 2 for vision_field = 7
        self.rng = np.random.RandomState(seed) if seed is not None else np.random
        self.reset()

    # ------------------------------------------------------------------ API
    def seed(self, seed: int) -> None:
        self.rng = np.random.RandomState(seed)

    def reset(self) -> dict[str, np.ndarray]:
        """Reset positions/episode state; returns the initial observation dict."""
        self.agent1_pos = [self.rng.randint(self.height - 1),
                           self.rng.randint(self.width - 1)]
        self.agent2_pos = [self.rng.randint(self.height - 1),
                           self.rng.randint(self.width - 1)]
        while self.agent1_pos == self.agent2_pos:
            self.agent1_pos = [self.rng.randint(self.height - 1),
                               self.rng.randint(self.width - 1)]
            self.agent2_pos = [self.rng.randint(self.height - 1),
                               self.rng.randint(self.width - 1)]

        # Rendering/analysis flags (toward = chaser closed distance,
        # away = explorer increased distance). Per-variant update rules live
        # in _move; they never influence rewards or events.
        self.toward = False
        self.away = False

        self.agent1_R = 0.0
        self.agent2_R = 0.0
        self.agent1_visited_fields = {tuple(self.agent1_pos)}
        self.agent2_visited_fields = {tuple(self.agent2_pos)}
        self.timesteps = 0

        # Previous-step discrete positions, used by the event bookkeeping.
        self.obs_hist = [self._get_discrete_obs()]

        # Per-episode metric counters (reference: callbacks_v1j).
        self.episode_metrics = dict(
            num_collisions=0, approaches=0,
            r1_new_fields=0, r2_new_fields=0,
            escapes_far=0, escapes_near=0, escapes_close=0,
            partner_visible_steps=0,
        )
        return self._get_obs()

    def step(self, actions) -> tuple[dict, dict, bool, dict]:
        """Advance one step. ``actions`` is a dict {"agent1": a1, "agent2": a2}
        or a pair (a1, a2). Returns (obs, rewards, done, info)."""
        if not isinstance(actions, dict):
            actions = {"agent1": int(actions[0]), "agent2": int(actions[1])}
        self.timesteps += 1
        is_done = self.timesteps >= self.timestep_limit

        # Agent2 (chaser) always moves first; both moves see the previous
        # step's discrete positions for event bookkeeping.
        prev_discrete = self.obs_hist[-1]
        events = self._move(self.agent2_pos, int(actions["agent2"]),
                            prev_discrete, is_agent1=False)
        events |= self._move(self.agent1_pos, int(actions["agent1"]),
                             prev_discrete, is_agent1=True)

        self.collision = "collision" in events
        self.R1_new_field = "agent1_new_field" in events
        self.R2_new_field = "agent2_new_field" in events
        self.approach = "approach" in events
        self.escape_far = "escape_far" in events
        self.escape_near = "escape_near" in events
        self.escape_close = "escape_close" in events

        obs = self._get_obs()
        self.obs_hist.append(self._get_discrete_obs())

        if self.social:
            r1 = -1.0 if "collision" in events else \
                1.0 if "agent1_new_field" in events else -0.5
            r2 = 1.0 if "collision" in events else \
                0.1 if "agent2_new_field" in events else -0.1
        else:
            r1 = -0.5 if "collision" in events else \
                1.0 if "agent1_new_field" in events else -0.5
            r2 = -0.1 if "collision" in events else \
                0.1 if "agent2_new_field" in events else -0.1
        self.agent1_R += r1
        self.agent2_R += r2
        self.r1, self.r2 = r1, r2

        m = self.episode_metrics
        m["num_collisions"] += int("collision" in events)
        m["approaches"] += int("approach" in events)
        m["r1_new_fields"] += int("agent1_new_field" in events)
        m["r2_new_fields"] += int("agent2_new_field" in events)
        m["escapes_far"] += int("escape_far" in events)
        m["escapes_near"] += int("escape_near" in events)
        m["escapes_close"] += int("escape_close" in events)
        m["partner_visible_steps"] += int(self.partner_in_fov())

        info = {
            "events": events,
            "agent1_pos": list(self.agent1_pos),
            "agent2_pos": list(self.agent2_pos),
            "partner_visible": self.partner_in_fov(),
        }
        if is_done:
            info["episode"] = {
                "r1": self.agent1_R, "r2": self.agent2_R, "l": self.timesteps,
                **self.episode_metrics,
            }
        return obs, {"agent1": r1, "agent2": r2}, is_done, info

    # -------------------------------------------------------------- internals
    def partner_in_fov(self) -> bool:
        """Whether the partner lies inside the 7x7 field of view (Chebyshev
        distance <= 3; symmetric, so the same for both agents)."""
        dr = abs(self.agent1_pos[0] - self.agent2_pos[0])
        dc = abs(self.agent1_pos[1] - self.agent2_pos[1])
        return dr <= self.vision_side and dc <= self.vision_side

    def _discrete_pos(self, pos) -> int:
        return pos[0] * self.width + (pos[1] % self.width)

    def _get_discrete_obs(self) -> dict[str, np.ndarray]:
        pair = np.array([self._discrete_pos(self.agent1_pos),
                         self._discrete_pos(self.agent2_pos)])
        return {"agent1": pair, "agent2": pair}

    def _get_obs(self) -> dict[str, np.ndarray]:
        """200-d observation per agent: [self one-hot (100) | partner one-hot
        (100)], the latter gated by ``partner_vision``."""
        a1_disc = self._discrete_pos(self.agent1_pos)
        a2_disc = self._discrete_pos(self.agent2_pos)
        n = self.width * self.height

        o1 = np.zeros(2 * n, dtype=np.float32)
        o1[a1_disc] = 1.0
        o2 = np.zeros(2 * n, dtype=np.float32)
        o2[a2_disc] = 1.0

        if self.partner_vision == "full" or (
                self.partner_vision == "fov7" and self.partner_in_fov()):
            o1[a2_disc + n] = 1.0
            o2[a1_disc + n] = 1.0
        return {"agent1": o1, "agent2": o2}

    def _prev_partner_coord(self, part_coord, is_agent1) -> list[int]:
        """Decode the partner's previous-step [row, col] from its discrete
        position (the reference decodes the two digits of a 0-99 integer,
        equivalent to divmod for a 10x10 grid)."""
        xx = int(part_coord["agent1"][1 if is_agent1 else 0])
        return [xx // self.width, xx % self.width]

    def _dist_flags(self, coords, orig_coords, prev_coord, is_agent1,
                    sticky: bool):
        """Update the toward/away rendering flags (no effect on events)."""
        new_dist = float(np.linalg.norm(np.array(coords) - np.array(prev_coord)))
        old_dist = float(np.linalg.norm(np.array(orig_coords) - np.array(prev_coord)))
        if is_agent1:
            if new_dist > old_dist:
                self.away = True
            elif not sticky:
                self.away = False
        else:
            if new_dist < old_dist:
                self.toward = True
            elif not sticky:
                self.toward = False
        return new_dist, old_dist

    def _move(self, coords, action, part_coord, is_agent1) -> set:
        """Apply one agent's move (mutates ``coords`` in place) and return the
        resulting event set."""
        orig_coords = coords[:]

        # Random-opponent arenas overwrite the action here.
        if (self.random_agent == 1 and is_agent1) or \
                (self.random_agent == 2 and not is_agent1):
            action = self.rng.randint(4)

        coords[0] += -1 if action == 0 else 1 if action == 2 else 0
        coords[1] += 1 if action == 1 else -1 if action == 3 else 0

        prev_coord = self._prev_partner_coord(part_coord, is_agent1)
        # Random-opponent arenas update the distance flags every move, before
        # the collision resolution.
        if self.random_agent is not None:
            self._dist_flags(coords, orig_coords, prev_coord, is_agent1,
                             sticky=False)

        # Collision: moving onto the other agent's current cell blocks the move.
        other = self.agent2_pos if is_agent1 else self.agent1_pos
        if coords == other:
            coords[0], coords[1] = orig_coords
            return {"collision"}

        # Walls clamp the position.
        coords[0] = min(max(coords[0], 0), self.height - 1)
        coords[1] = min(max(coords[1], 0), self.width - 1)

        visited = self.agent1_visited_fields if is_agent1 \
            else self.agent2_visited_fields
        if tuple(coords) not in visited:
            visited.add(tuple(coords))
            if self.random_agent is None and not self.social:
                self._dist_flags(coords, orig_coords, prev_coord, is_agent1,
                                 sticky=True)
            return {"agent1_new_field" if is_agent1 else "agent2_new_field"}

        thr1, thr2 = 5, 3
        new_dist = float(np.linalg.norm(np.array(coords) - np.array(prev_coord)))
        old_dist = float(np.linalg.norm(np.array(orig_coords) - np.array(prev_coord)))
        if is_agent1:
            if new_dist >= thr1 and new_dist > old_dist:
                if self.random_agent is None and not self.social:
                    self.away = True
                return {"escape_far"}
            elif new_dist >= thr2 and new_dist > old_dist:
                if self.random_agent is None and not self.social:
                    self.away = True
                return {"escape_near"}
            elif thr2 > new_dist > old_dist:
                if self.random_agent is None and not self.social:
                    self.away = True
                return {"escape_close"}
        else:
            if new_dist < old_dist:
                if self.random_agent is None and not self.social:
                    self.toward = True
                return {"approach"}
        return set()

    # -------------------------------------------------------------- rendering
    def render_ascii(self) -> str:
        lines = ["_" * (self.width + 2)]
        for r in range(self.height):
            row = "|"
            for c in range(self.width):
                if self.agent1_pos == [r, c]:
                    row += "1"
                elif self.agent2_pos == [r, c]:
                    row += "2"
                elif (r, c) in self.agent1_visited_fields:
                    row += "."
                else:
                    row += " "
            lines.append(row + "|")
        lines.append("~" * (self.width + 2))
        lines.append(f"R1={self.agent1_R: .1f} R2={self.agent2_R: .1f} "
                     f"t={self.timesteps}")
        return "\n".join(lines)


def make_arena(condition: str = "social", episode_len: int = 100,
               random_agent: int | None = None, seed: int | None = None,
               **kwargs) -> ChaseArena:
    """Convenience factory.

    condition: "social" (v1d_5), "nonsocial" (v1d_11),
        "nonsocial_fov7" (v1d_11_2), "nonsocial_fullvision" (v1d_11_3).
    random_agent: None / 1 / 2 for the random-opponent playoff arenas.
    """
    table = {
        "social": dict(social=True, partner_vision="fov7"),
        "nonsocial": dict(social=False, partner_vision="none"),
        "nonsocial_fov7": dict(social=False, partner_vision="fov7"),
        "nonsocial_fullvision": dict(social=False, partner_vision="full"),
    }
    if condition not in table:
        raise KeyError(f"unknown condition '{condition}'; pick from {sorted(table)}")
    return ChaseArena(episode_len=episode_len, random_agent=random_agent,
                      seed=seed, **table[condition], **kwargs)


class SyncVectorChaseEnv:
    """Synchronous vector wrapper over N chase arenas (pairs of agents).

    Mirrors the semantics of ``neuralrnn.rl.envs.base.SyncVectorEnv`` but for
    the two-agent arena: ``step`` takes per-env action pairs and returns
    per-agent observations and rewards. Episodes auto-reset on the fixed
    horizon; the terminal step's metrics are in ``info['episode']`` and the
    final observation in ``info['final_observation']``.
    """

    def __init__(self, env_fns):
        self.envs = [fn() for fn in env_fns]
        self.num_envs = len(self.envs)
        self.single_obs_dim = OBS_DIM
        self.single_num_actions = N_ACTIONS

    def reset(self, seed: int | None = None):
        o1, o2 = [], []
        for i, env in enumerate(self.envs):
            if seed is not None:
                env.seed(seed + i)
            obs = env.reset()
            o1.append(obs["agent1"])
            o2.append(obs["agent2"])
        return np.stack(o1), np.stack(o2)

    def step(self, actions1, actions2):
        """actions1/actions2: (N,) integer arrays. Returns
        (obs1, obs2): (N, 200) float32; (r1, r2, dones): (N,); infos: list."""
        o1, o2, r1, r2, dones, infos = [], [], [], [], [], []
        for env, a1, a2 in zip(self.envs, actions1, actions2):
            obs, rew, done, info = env.step({"agent1": int(a1),
                                             "agent2": int(a2)})
            if done:
                info = dict(info)
                info["final_observation"] = (obs["agent1"].copy(),
                                             obs["agent2"].copy())
                obs = env.reset()
            o1.append(obs["agent1"])
            o2.append(obs["agent2"])
            r1.append(rew["agent1"])
            r2.append(rew["agent2"])
            dones.append(done)
            infos.append(info)
        return (np.stack(o1), np.stack(o2),
                np.asarray(r1, dtype=np.float32), np.asarray(r2, dtype=np.float32),
                np.asarray(dones, dtype=bool), infos)

### Watching an episode

Before training anything, let us look at the task itself. The panels below roll out two untrained agents: trajectories on the grid, the events of each step, and whether the chaser can see the explorer.

In [ ]:
def plot_episode_panel(ax, env, pos1_hist, pos2_hist, title=""):
    """Draw the grid with both agents' trajectories."""
    ax.set_xlim(-0.5, env.width - 0.5)
    ax.set_ylim(env.height - 0.5, -0.5)
    ax.set_xticks(range(env.width)); ax.set_yticks(range(env.height))
    ax.grid(True, color="0.9", lw=0.5)
    ax.set_aspect("equal")
    p1, p2 = np.asarray(pos1_hist), np.asarray(pos2_hist)
    ax.plot(p1[:, 1], p1[:, 0], "-", color="#1f77b4", lw=1.2, alpha=0.8,
            label="explorer")
    ax.plot(p2[:, 1], p2[:, 0], "-", color="#d62728", lw=1.2, alpha=0.8,
            label="chaser")
    ax.plot(p1[0, 1], p1[0, 0], "o", color="#1f77b4", ms=7, mec="k")
    ax.plot(p2[0, 1], p2[0, 0], "o", color="#d62728", ms=7, mec="k")
    ax.plot(p1[-1, 1], p1[-1, 0], "s", color="#1f77b4", ms=7, mec="k")
    ax.plot(p2[-1, 1], p2[-1, 0], "s", color="#d62728", ms=7, mec="k")
    ax.set_title(title)


def rollout_episode(env, policy1=None, policy2=None, seed=0):
    """Roll out one episode with random or provided policies; return records."""
    rng = np.random.RandomState(seed)
    obs = env.reset()
    pos1_hist = [list(env.agent1_pos)]
    pos2_hist = [list(env.agent2_pos)]
    events_hist, visible_hist = [], []
    for _ in range(env.timestep_limit):
        a1 = rng.randint(4) if policy1 is None else policy1(obs["agent1"])
        a2 = rng.randint(4) if policy2 is None else policy2(obs["agent2"])
        obs, rew, done, info = env.step({"agent1": a1, "agent2": a2})
        pos1_hist.append(list(env.agent1_pos))
        pos2_hist.append(list(env.agent2_pos))
        events_hist.append(info["events"])
        visible_hist.append(info["partner_visible"])
    return pos1_hist, pos2_hist, events_hist, visible_hist


env = make_arena("social", episode_len=100, seed=3)
pos1_hist, pos2_hist, events_hist, visible_hist = rollout_episode(env, seed=3)

fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.4),
                         gridspec_kw={"width_ratios": [1, 1.6]})
plot_episode_panel(axes[0], env, pos1_hist, pos2_hist,
                   "untrained agents (random walk)")
axes[0].legend(loc="upper right", fontsize=8)

# event timeline: one colored band per agent
for t, ev in enumerate(events_hist):
    c1 = ("collision" if "collision" in ev else
          "new_field" if "agent1_new_field" in ev else
          "escape_far" if "escape_far" in ev else
          "escape_near" if "escape_near" in ev else
          "escape_close" if "escape_close" in ev else "none")
    c2 = ("collision" if "collision" in ev else
          "new_field" if "agent2_new_field" in ev else
          "approach" if "approach" in ev else "none")
    axes[1].axvspan(t, t + 1, 0.55, 1.0, color=EVENT_COLORS[c1])
    axes[1].axvspan(t, t + 1, 0.05, 0.5, color=EVENT_COLORS[c2])
axes[1].set_xlim(0, env.timestep_limit); axes[1].set_ylim(0, 1)
axes[1].set_yticks([0.275, 0.775]); axes[1].set_yticklabels(["chaser", "explorer"])
axes[1].set_xlabel("time step")
handles = [plt.Rectangle((0, 0), 1, 1, color=EVENT_COLORS[k]) for k in
           ["new_field", "collision", "approach", "escape_far",
            "escape_near", "escape_close"]]
axes[1].legend(handles, ["new field", "collision", "approach", "escape (far)",
                         "escape (near)", "escape (close)"],
               ncol=3, fontsize=7.5, loc="upper center",
               bbox_to_anchor=(0.5, 1.22))
axes[1].set_title("events per step")
fig.tight_layout()
savefig(fig, "fig_task_example.png")
plt.show()

print(f"chaser saw the explorer on {np.mean(visible_hist):.0%} of steps")

## The agents

Each agent is a vanilla ReLU RNN with 256 hidden units, reading the 200-dimensional observation and emitting both a policy and a value:

$$h_t = \mathrm{ReLU}\!\left(W_{in}\, x_t + b_{in} + W_{rec}\, h_{t-1} + b_{rec}\right), \qquad \pi_\theta(a \mid h_t) = \mathrm{softmax}\!\left(W_{out}\, h_t + b_{out}\right), \qquad V_\theta(h_t) = w_v^\top h_t + b_v .$$

In the framework this is an `ActorCriticModel` whose core is a `CTRNN` configured with Euler scaling $\alpha = 1$ (i.e. the continuous-time dynamics evaluated with $dt = \tau$, recovering exactly the discrete-time equation above, including the two separate biases of `nn.RNN`). The hidden state is initialized to zero at the start of every episode. The two agents are **fully independent networks** — no shared weights, no shared observations beyond what the FOV reveals — mirroring the two mice of the study.

In [ ]:
import torch

from neuralrnn.models.actor_critic import ActorCriticConfig, ActorCriticModel
from neuralrnn.models.ctrnn import CTRNNConfig

OBS_DIM = 200
HIDDEN_DIM = 256
N_ACTIONS = 4


def make_chase_agent(seed: int | None = None) -> ActorCriticModel:
    """Build one chase agent (explorer or chaser — same architecture)."""
    if seed is not None:
        torch.manual_seed(seed)
    core = CTRNNConfig(
        input_dim=OBS_DIM,
        latent_dim=HIDDEN_DIM,
        output_dim=N_ACTIONS,
        alpha=1.0,          # discrete-time RNN: h_t = f(W_in x + W_rec h + b)
        activation="relu",
    )
    config = ActorCriticConfig(
        core_config=core.to_dict(),
        action_dim=N_ACTIONS,
        action_type="discrete",
        policy_logit_scale=1.0,
    )
    return ActorCriticModel(config)


def make_agent_pair(seed: int = 0) -> tuple[ActorCriticModel, ActorCriticModel]:
    """Build the (explorer, chaser) pair with independent weight draws."""
    return make_chase_agent(seed=2 * seed), make_chase_agent(seed=2 * seed + 1)

# build one untrained pair and count parameters
agent1, agent2 = make_agent_pair(seed=SEED)
n_params = sum(p.numel() for p in agent1.parameters())
print(f"each agent: {n_params:,} parameters "
      f"(W_in {200}x{256}, W_rec {256}x{256}, actor {256}x4, critic {256}x1)")
print(agent1.config.model_type, "core:", agent1.core.config.model_type)

Two properties of this architecture matter for what follows. First, the policy is *linear* in the hidden state, so any computation the agent needs for action selection must be linearly readable from $h_t$ — this is what makes the neural-action-space and perturbation analyses below meaningful. Second, the only channel through which one agent can influence the other's neural activity is the partner one-hot in the observation: any correlation between the two networks' activity that cannot be explained by shared self-related structure is evidence of a genuinely *social* representation.

## Training with multi-agent PPO

Both agents are trained simultaneously with independent **PPO** updates — each agent treats the other as part of its environment. The recipe follows the reference implementation exactly (verified against its RLlib 2.2 configuration and source):

- **Rollout.** Each iteration runs 40 parallel arenas for one full 100-step episode each (4000 environment steps per agent). Because episodes are aligned with the rollout, no done-masking is needed; the fixed horizon is terminal, so the value target of the last step is just its reward.
- **Advantage.** GAE with $\lambda = 1$ and $\gamma = 0.99$, i.e. $\lambda$-returns: $\hat A_t = \sum_{\ell \ge 0} \gamma^{\ell} r_{t+\ell} - V(h_t)$.
- **Objective** (per agent, per minibatch):

$$\mathcal{L} = \underbrace{-\mathbb{E}\!\left[\min\!\big(\rho_t \hat A_t,\; \mathrm{clip}(\rho_t, 0.7, 1.3)\, \hat A_t\big)\right]}_{\text{clipped surrogate},\; \rho_t = \pi_\theta/\pi_{\theta_{\text{old}}}} + \underbrace{\mathbb{E}\!\left[\mathrm{clamp}\!\big((V_\theta(h_t) - R_t)^2, 0, 10\big)\right]}_{\text{clipped value loss}} + \underbrace{\beta\, \mathbb{E}\!\left[\mathrm{KL}\!\big(\pi_{\theta_{\text{old}}} \,\|\, \pi_\theta\big)\right]}_{\text{exact KL penalty}} + \; \lambda_{rec}\, \|W_{rec}\|_2$$

  with $\beta$ adapted after every iteration (doubled-ish $\times 1.5$ when the mean KL exceeds $2\times$ the target 0.01, halved below $0.5\times$) and an **unsquared** $\ell_2$ penalty $\lambda_{rec} = 3.0$ on the recurrent weights only. The entropy bonus is zero — exploration comes from the large batch and the KL trust region.
- **Optimization.** Backpropagation through time in chunks of 20 steps starting from the hidden states stored during the rollout; 30 epochs per iteration over minibatches of 128 chunks; Adam, learning rate $5\times10^{-5}$, no gradient clipping.

The trainer below reuses the framework's `GAE` and the `ActorCriticModel` interface (`step`, `evaluate_sequence`, `readout`, `init_state`); the framework `RLTrainer` itself is single-agent, so the two-agent loop lives here. Checkpoints are written with `save_pretrained`, so trained pairs load back with plain `AutoModel.from_pretrained`.

In [ ]:
import json
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import torch
from torch.distributions import Categorical

from neuralrnn.rl.returns import GAE


@dataclass
class ChasePPOConfig:
    num_envs: int = 40
    episode_len: int = 100
    total_iterations: int = 8000
    gamma: float = 0.99
    gae_lambda: float = 1.0
    lr: float = 5e-5
    clip_param: float = 0.3
    kl_coeff_init: float = 0.2
    kl_target: float = 0.01
    vf_clip_param: float = 10.0
    chunk_len: int = 20
    sgd_minibatch_size: int = 128   # in BPTT chunks
    num_sgd_iter: int = 30
    l2_rec: float = 3.0
    l2_inp: float = 0.0
    device: str = "cpu"
    seed: int = 0
    checkpoint_every: int = 200
    log_every: int = 10


class _AgentBatch:
    """Rollout storage for one agent: time-first (T, N, ...) tensors."""

    def __init__(self, T: int, N: int, obs_dim: int, n_actions: int,
                 hidden_dim: int, device):
        self.T, self.N = T, N
        self.obs = torch.zeros(T, N, obs_dim, device=device)
        self.actions = torch.zeros(T, N, dtype=torch.long, device=device)
        self.logprobs = torch.zeros(T, N, device=device)
        self.logits = torch.zeros(T, N, n_actions, device=device)
        self.values = torch.zeros(T, N, device=device)
        self.rewards = torch.zeros(T, N, device=device)
        self.states = torch.zeros(T, N, hidden_dim, device=device)


class ChasePPOTrainer:
    """Train the (explorer, chaser) pair with independent PPO updates."""

    def __init__(self, agent1, agent2, envs, config: ChasePPOConfig,
                 out_dir: str | Path):
        self.cfg = config
        self.device = torch.device(config.device)
        self.agent1 = agent1.to(self.device)
        self.agent2 = agent2.to(self.device)
        self.envs = envs
        assert envs.num_envs == config.num_envs
        assert config.episode_len % config.chunk_len == 0
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.opt1 = torch.optim.Adam(self.agent1.parameters(), lr=config.lr)
        self.opt2 = torch.optim.Adam(self.agent2.parameters(), lr=config.lr)
        self.gae = GAE(gae_lambda=config.gae_lambda)
        self.kl_coeff = {"agent1": config.kl_coeff_init,
                         "agent2": config.kl_coeff_init}
        self.rng = np.random.RandomState(config.seed)
        self.history: list[dict] = []

    # ------------------------------------------------------------------ main
    def train(self, start_iteration: int = 0) -> list[dict]:
        cfg = self.cfg
        for it in range(start_iteration, cfg.total_iterations):
            t0 = time.time()
            b1, b2, ep_metrics = self._collect_rollout()
            stats = self._update(b1, b2)
            rec = {
                "iteration": it + 1,
                "global_steps": (it + 1) * cfg.num_envs * cfg.episode_len,
                **{f"episode/{k}": float(np.mean([m[k] for m in ep_metrics]))
                   for k in ep_metrics[0]},
                **{f"loss/{k}": v for k, v in stats.items()},
                "kl_coeff/agent1": self.kl_coeff["agent1"],
                "kl_coeff/agent2": self.kl_coeff["agent2"],
                "sec_per_iter": time.time() - t0,
            }
            self.history.append(rec)
            self._dump_history()
            if (it + 1) % cfg.log_every == 0:
                print(f"iter {it + 1}/{cfg.total_iterations} "
                      f"R1={rec['episode/r1']:+.2f} R2={rec['episode/r2']:+.2f} "
                      f"coll={rec['episode/num_collisions']:.2f} "
                      f"kl={stats['kl/agent1']:.4f}/{stats['kl/agent2']:.4f} "
                      f"({rec['sec_per_iter']:.1f}s)", flush=True)
            if (it + 1) % cfg.checkpoint_every == 0:
                self.save_checkpoint(f"checkpoint-iter{it + 1}")
        self.save_checkpoint("final")
        return self.history

    # --------------------------------------------------------------- rollout
    def _collect_rollout(self):
        """Run ``episode_len`` steps in all arenas (episode-aligned: arenas are
        reset at the start so no done-masking is needed mid-rollout)."""
        cfg = self.cfg
        T, N = cfg.episode_len, cfg.num_envs
        dev = self.device
        self.agent1.eval()
        self.agent2.eval()

        o1, o2 = self.envs.reset()
        b1 = _AgentBatch(T, N, self.envs.single_obs_dim,
                         self.envs.single_num_actions,
                         self.agent1.config.latent_dim, dev)
        b2 = _AgentBatch(T, N, self.envs.single_obs_dim,
                         self.envs.single_num_actions,
                         self.agent2.config.latent_dim, dev)
        z1 = self.agent1.init_state(N, dev)
        z2 = self.agent2.init_state(N, dev)

        ep_metrics = []
        with torch.no_grad():
            for t in range(T):
                x1 = torch.as_tensor(o1, dtype=torch.float32, device=dev)
                x2 = torch.as_tensor(o2, dtype=torch.float32, device=dev)
                a1, logp1, _, v1, z1 = self.agent1.step(x1, z1)
                a2, logp2, _, v2, z2 = self.agent2.step(x2, z2)
                b1.obs[t], b2.obs[t] = x1, x2
                b1.actions[t], b2.actions[t] = a1, a2
                b1.logprobs[t], b2.logprobs[t] = logp1, logp2
                b1.logits[t] = self.agent1.readout(z1)
                b2.logits[t] = self.agent2.readout(z2)
                b1.values[t], b2.values[t] = v1, v2
                b1.states[t], b2.states[t] = z1, z2

                (o1, o2, r1, r2, dones, infos) = self.envs.step(
                    a1.cpu().numpy(), a2.cpu().numpy())
                b1.rewards[t] = torch.as_tensor(r1, device=dev)
                b2.rewards[t] = torch.as_tensor(r2, device=dev)
                for info in infos:
                    if "episode" in info:
                        ep_metrics.append(info["episode"])

        # Terminal bootstrap: the fixed horizon ends the episode, so the
        # value target of the last step is just its reward.
        zeros = torch.zeros(N, device=dev)
        ones = torch.ones(N, device=dev)
        dones_zero = torch.zeros(T, N, device=dev)
        for b in (b1, b2):
            b.returns, b.advantages = self.gae.compute(
                b.rewards, b.values, dones_zero, zeros, ones, cfg.gamma)
        return b1, b2, ep_metrics

    # ---------------------------------------------------------------- update
    def _agent_loss(self, agent, batch, kl_coeff):
        """RLlib-2.2 PPO total loss for one minibatch (see module docstring)."""
        new_logp, _, new_values, states, _ = agent.evaluate_sequence(
            batch["obs"], batch["z0"], batch["dones"], batch["actions"])
        # Logits of the current policy from the replayed states (readout
        # returns scaled policy logits for discrete agents).
        new_logits = agent.readout(states)

        logp_ratio = torch.exp(new_logp - batch["logprobs"])
        adv = batch["advantages"]
        surrogate = torch.min(
            adv * logp_ratio,
            adv * torch.clamp(logp_ratio, 1 - self.cfg.clip_param,
                              1 + self.cfg.clip_param))
        vf_loss = torch.clamp((new_values - batch["returns"]) ** 2,
                              0, self.cfg.vf_clip_param)
        prev_dist = Categorical(logits=batch["logits"])
        curr_dist = Categorical(logits=new_logits)
        kl = torch.distributions.kl_divergence(prev_dist, curr_dist)

        loss = (-surrogate + vf_loss).mean() + kl_coeff * kl.mean()
        l2 = self.cfg.l2_rec * torch.norm(agent.core.h2h.weight) \
            + self.cfg.l2_inp * torch.norm(agent.core.input2h.weight)
        loss = loss + l2
        logs = {"kl": float(kl.mean().detach()),
                "pg": float((-surrogate.mean()).detach()),
                "vf": float(vf_loss.mean().detach()),
                "l2": float(l2.detach())}
        return loss, logs

    def _update(self, b1, b2):
        cfg = self.cfg
        self.agent1.train()
        self.agent2.train()
        T, N = cfg.episode_len, cfg.num_envs
        n_chunks = T // cfg.chunk_len
        chunk_idx = [(e, c) for e in range(N) for c in range(n_chunks)]
        dev = self.device

        def gather(b, sel):
            obs = torch.stack([b.obs[c * cfg.chunk_len:(c + 1) * cfg.chunk_len, e]
                               for e, c in sel])
            return {
                "obs": obs,
                "actions": torch.stack(
                    [b.actions[c * cfg.chunk_len:(c + 1) * cfg.chunk_len, e]
                     for e, c in sel]),
                "logprobs": torch.stack(
                    [b.logprobs[c * cfg.chunk_len:(c + 1) * cfg.chunk_len, e]
                     for e, c in sel]),
                "logits": torch.stack(
                    [b.logits[c * cfg.chunk_len:(c + 1) * cfg.chunk_len, e]
                     for e, c in sel]),
                "advantages": torch.stack(
                    [b.advantages[c * cfg.chunk_len:(c + 1) * cfg.chunk_len, e]
                     for e, c in sel]),
                "returns": torch.stack(
                    [b.returns[c * cfg.chunk_len:(c + 1) * cfg.chunk_len, e]
                     for e, c in sel]),
                "z0": torch.stack([b.states[c * cfg.chunk_len, e]
                                   for e, c in sel]).detach(),
                # Chunks never cross episode boundaries (rollout is
                # episode-aligned), so no mid-chunk resets.
                "dones": torch.zeros(len(sel), cfg.chunk_len, device=dev),
            }

        kl_sum = {"agent1": 0.0, "agent2": 0.0}
        n_mb = 0
        for _ in range(cfg.num_sgd_iter):
            self.rng.shuffle(chunk_idx)
            for start in range(0, len(chunk_idx), cfg.sgd_minibatch_size):
                sel = chunk_idx[start:start + cfg.sgd_minibatch_size]
                for b, agent, opt, name in (
                        (b1, self.agent1, self.opt1, "agent1"),
                        (b2, self.agent2, self.opt2, "agent2")):
                    loss, logs = self._agent_loss(agent, gather(b, sel),
                                                  self.kl_coeff[name])
                    opt.zero_grad()
                    loss.backward()
                    opt.step()
                    kl_sum[name] += logs["kl"]
                n_mb += 1
        for name in kl_sum:
            mean_kl = kl_sum[name] / n_mb
            if mean_kl > 2.0 * cfg.kl_target:
                self.kl_coeff[name] *= 1.5
            elif mean_kl < 0.5 * cfg.kl_target:
                self.kl_coeff[name] *= 0.5
        return {"kl/agent1": kl_sum["agent1"] / n_mb,
                "kl/agent2": kl_sum["agent2"] / n_mb}

    # ------------------------------------------------------------ checkpoints
    def save_checkpoint(self, name: str) -> None:
        dest = self.out_dir / name
        self.agent1.save_pretrained(dest / "agent1")
        self.agent2.save_pretrained(dest / "agent2")
        with open(dest / "trainer_state.json", "w") as f:
            json.dump({"kl_coeff": self.kl_coeff,
                       "config": asdict(self.cfg)}, f, indent=2)

    def _dump_history(self) -> None:
        with open(self.out_dir / "history.json", "w") as f:
            json.dump(self.history, f)

    def load_checkpoint(self, path: str | Path) -> None:
        """Resume weights + trainer state from a checkpoint directory."""
        from neuralrnn import AutoModel
        path = Path(path)
        self.agent1.load_state_dict(
            AutoModel.from_pretrained(path / "agent1").state_dict())
        self.agent2.load_state_dict(
            AutoModel.from_pretrained(path / "agent2").state_dict())
        with open(path / "trainer_state.json") as f:
            state = json.load(f)
        self.kl_coeff = {k: float(v) for k, v in state["kl_coeff"].items()}

In [ ]:
# Full training run, disabled by default (see the note below): set
# RUN_TRAINING = True to train here, or copy this cell and the trainer cell
# above into a standalone .py file and run it as a background job.
RUN_TRAINING = False


def train_pair(condition, seed=0, iters=8000, device=DEVICE,
               init_from=None, start_iteration=0):
    """Train one (explorer, chaser) pair on one condition.

    Checkpoints, per-iteration ``history.json`` and ``run_meta.json`` (written
    last; the load-first cells below take it as the completion flag) go to
    ``PAIR_DIRS[condition]``. Resume an interrupted run with ``init_from``
    (a checkpoint directory) and ``start_iteration``.
    """
    out_dir = PAIR_DIRS[condition]
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(seed)
    np.random.seed(seed)

    cfg = ChasePPOConfig(total_iterations=iters, device=device, seed=seed)
    agent1, agent2 = make_agent_pair(seed=seed)
    # 40 parallel arenas with distinct seeds, episode-aligned rollouts.
    envs = SyncVectorChaseEnv([
        (lambda i=i: make_arena(condition, episode_len=cfg.episode_len,
                                seed=seed * 100003 + i))
        for i in range(cfg.num_envs)])

    trainer = ChasePPOTrainer(agent1, agent2, envs, cfg, out_dir)
    if init_from is not None:
        trainer.load_checkpoint(init_from)
        print(f"resumed weights from {init_from}")

    t0 = time.time()
    trainer.train(start_iteration=start_iteration)
    trainer.save_checkpoint("final")
    with open(out_dir / "run_meta.json", "w") as f:  # completion flag, last
        json.dump({"condition": condition, "seed": seed, "iterations": iters,
                   "device": device,
                   "wall_hours": (time.time() - t0) / 3600}, f, indent=2)
    print(f"done in {(time.time() - t0) / 3600:.2f} h -> {out_dir}")


if RUN_TRAINING:
    train_pair("social", seed=SEED)
    train_pair("nonsocial", seed=SEED)


The two cells above are the complete training code: the `ChasePPOTrainer` class and the `train_pair` driver that wires it to the arena and the agent pair. Training one pair (8000 iterations, roughly half a day on a single GPU) is too long for an interactive session, so the driver is guarded by `RUN_TRAINING = False` — flip it to train in the notebook, or lift the same code out and run it as a standalone Python job. A ready-made CLI wrapper with resume support lives at `environment_chase_code/train_pair.py`:

```bash
python -m environment_chase_code.train_pair --condition social    --seed 0 --iters 8000 --device cuda
python -m environment_chase_code.train_pair --condition nonsocial --seed 0 --iters 8000 --device cuda
```

The cells below are **load-first**: if a finished run (`run_meta.json`) is present we load it; otherwise we say so and skip.

In [ ]:
def load_pair(condition):
    """Load the trained (explorer, chaser) pair of a condition, or None."""
    d = PAIR_DIRS[condition] / "final"
    if (d / "agent1").exists():
        a1 = AutoModel.from_pretrained(d / "agent1")
        a2 = AutoModel.from_pretrained(d / "agent2")
        print(f"[{condition}] loaded {d}")
        return a1, a2
    print(f"[{condition}] no trained pair under {PAIR_DIRS[condition]} — "
          f"run the training cell above first.")
    return None


def load_history(condition):
    f = PAIR_DIRS[condition] / "history.json"
    return json.load(open(f)) if f.exists() else None


pairs = {c: load_pair(c) for c in ("social", "nonsocial")}
histories = {c: load_history(c) for c in ("social", "nonsocial")}

### Learning curves

How the two agents' behavior develops over training: the rewards of chaser and explorer, the two *social* metrics (collisions, and the fraction of time the chaser keeps the explorer in its field of vision), and the two *non-social* metrics (new fields explored by each agent). This is the analog of Fig. 5c–h of the paper.

In [ ]:
def smooth(y, w=25):
    """Centered moving average with edge padding."""
    y = np.asarray(y, dtype=float)
    k = np.ones(w) / w
    return np.convolve(np.pad(y, (w // 2, w - 1 - w // 2), mode="edge"),
                       k, mode="valid")


def plot_training_curves(histories):
    fig, axes = plt.subplots(2, 3, figsize=(10.5, 5.2))
    panels = [
        ("episode/r2", "chaser reward / episode"),
        ("episode/r1", "explorer reward / episode"),
        ("episode/num_collisions", "collisions / episode"),
        ("episode/partner_visible_steps", "steps with partner in vision"),
        ("episode/r2_new_fields", "chaser new fields / episode"),
        ("episode/r1_new_fields", "explorer new fields / episode"),
    ]
    for ax, (key, label) in zip(axes.flat, panels):
        for cond, color in (("social", COLOR_SOCIAL),
                            ("nonsocial", COLOR_NONSOCIAL)):
            h = histories.get(cond)
            if h is None or key not in h[0]:
                continue
            x = [r["iteration"] for r in h]
            y = [r[key] for r in h]
            ax.plot(x, y, color=color, alpha=0.15, lw=0.6)
            ax.plot(x, smooth(y), color=color, lw=1.6, label=cond)
        ax.set_xlabel("training iteration")
        ax.set_ylabel(label)
        ax.legend(fontsize=8)
    fig.suptitle("learning curves, social vs non-social training", y=1.0)
    fig.tight_layout()
    return fig


if any(histories.values()):
    fig = plot_training_curves(histories)
    savefig(fig, "fig_training_curves.png")
    plt.show()
else:
    print("no training histories found — run the training cell "
          "above first.")

In the social condition the chaser's reward and the collision count climb together, and the chaser quickly learns to keep the explorer inside its $7\times7$ field of vision, while the explorer's new-field count stalls as it is forced to keep moving. In the non-social condition collisions stay at the chance level of two independent random walkers and both agents maximize their own exploration instead. Note the explorer's reward stays strongly positive throughout in both conditions — visiting new fields pays $+1$ per step and dominates the occasional $-1$ collision penalty.

### Watching the trained agents

What does the learned interaction actually look like? Below we roll out the **trained social pair** for one 200-step episode (actions are sampled from the policies, as during training). The chaser (red) finds the explorer (blue), locks it inside its field of vision, and collides with it over and over, while the explorer keeps darting away to break line of sight — the mouse-chase dynamics of Fig. 5b of the paper, emerging here from two independent PPO updates with no shared parameters.

In [ ]:
def rollout_trained_episode(a1, a2, condition="social", episode_len=200,
                            seed=4):
    """One episode of a trained pair (sampled actions); returns position,
    event and partner-visibility traces for plotting and animation."""
    env = make_arena(condition, episode_len=episode_len, seed=seed)
    obs = env.reset()
    z1 = a1.init_state(1, DEVICE)
    z2 = a2.init_state(1, DEVICE)
    pos1_hist = [list(env.agent1_pos)]
    pos2_hist = [list(env.agent2_pos)]
    events_hist, visible_hist = [], []
    with torch.no_grad():
        for _ in range(episode_len):
            x1 = torch.as_tensor(obs["agent1"], dtype=torch.float32,
                                 device=DEVICE).unsqueeze(0)
            x2 = torch.as_tensor(obs["agent2"], dtype=torch.float32,
                                 device=DEVICE).unsqueeze(0)
            z1 = a1.core.recurrence(x1, z1)
            z2 = a2.core.recurrence(x2, z2)
            act1 = int(Categorical(logits=a1.actor(z1)).sample())
            act2 = int(Categorical(logits=a2.actor(z2)).sample())
            obs, rew, done, info = env.step({"agent1": act1, "agent2": act2})
            pos1_hist.append(list(env.agent1_pos))
            pos2_hist.append(list(env.agent2_pos))
            events_hist.append(info["events"])
            visible_hist.append(info["partner_visible"])
    return pos1_hist, pos2_hist, events_hist, visible_hist


def episode_animation_html(pos1_hist, pos2_hist, fov=3, interval=120):
    """HTML animation of an episode; the chaser's FOV is shaded."""
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML
    p1, p2 = np.asarray(pos1_hist), np.asarray(pos2_hist)
    T = len(p1)
    fig, ax = plt.subplots(figsize=(4.2, 4.2),  dpi=80)
    ax.set_xlim(-0.5, 9.5); ax.set_ylim(9.5, -0.5)
    # ax.axis('off')
    # ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.grid(True, color="0.9", lw=0.5); ax.set_aspect("equal")
    fov_patch = plt.Rectangle((0, 0), 2 * fov + 1, 2 * fov + 1,
                              color="red", alpha=0.08, zorder=0)
    ax.add_patch(fov_patch)
    trail1, = ax.plot([], [], "-", color="#1f77b4", lw=1.4)
    trail2, = ax.plot([], [], "-", color="#d62728", lw=1.4)
    dot1, = ax.plot([], [], "o", color="#1f77b4", ms=9, mec="k")
    dot2, = ax.plot([], [], "o", color="#d62728", ms=9, mec="k")
    title = ax.set_title("")

    def update(t):
        trail1.set_data(p1[:t + 1, 1], p1[:t + 1, 0])
        trail2.set_data(p2[:t + 1, 1], p2[:t + 1, 0])
        dot1.set_data([p1[t, 1]], [p1[t, 0]])
        dot2.set_data([p2[t, 1]], [p2[t, 0]])
        fov_patch.set_xy((p2[t, 1] - fov - 0.5, p2[t, 0] - fov - 0.5))
        vis = (abs(p1[t, 0] - p2[t, 0]) <= fov) and (abs(p1[t, 1] - p2[t, 1]) <= fov)
        # title.set_text(f"step {t} — explorer {'in' if vis else 'OUT of'} chaser's vision")
        return trail1, trail2, dot1, dot2, fov_patch, title

    anim = FuncAnimation(fig, update, frames=T, interval=interval, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


anim_html = None
if pairs["social"] is not None:
    a1, a2 = pairs["social"]
    a1, a2 = a1.to(DEVICE).eval(), a2.to(DEVICE).eval()
    tpos1, tpos2, tevents, tvis = rollout_trained_episode(a1, a2)
    n_coll = sum("collision" in ev for ev in tevents)

    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.4),
                             gridspec_kw={"width_ratios": [1, 1.6]})
    plot_episode_panel(axes[0], make_arena("social"), tpos1, tpos2,
                       f"trained pair, 200 steps ({n_coll} collisions)")
    axes[0].legend(loc="upper right", fontsize=8)
    for t, ev in enumerate(tevents):
        c1 = ("collision" if "collision" in ev else
              "new_field" if "agent1_new_field" in ev else
              "escape_far" if "escape_far" in ev else
              "escape_near" if "escape_near" in ev else
              "escape_close" if "escape_close" in ev else "none")
        c2 = ("collision" if "collision" in ev else
              "new_field" if "agent2_new_field" in ev else
              "approach" if "approach" in ev else "none")
        axes[1].axvspan(t, t + 1, 0.55, 1.0, color=EVENT_COLORS[c1])
        axes[1].axvspan(t, t + 1, 0.05, 0.5, color=EVENT_COLORS[c2])
    axes[1].set_xlim(0, len(tevents)); axes[1].set_ylim(0, 1)
    axes[1].set_yticks([0.275, 0.775])
    axes[1].set_yticklabels(["chaser", "explorer"])
    axes[1].set_xlabel("time step")
    axes[1].set_title("events per step")
    fig.tight_layout()
    savefig(fig, "fig_trained_episode.png")
    plt.show()
    anim_html = episode_animation_html(tpos1[:80], tpos2[:80])
    print(f"trained episode: {n_coll} collisions, explorer in the chaser's "
          f"vision {np.mean(tvis):.0%} of steps")
else:
    print("no trained social pair found — run the training cell above first.")

In [ ]:
anim_html

Compare with the random-walk episode at the top of the notebook: where the untrained pair wandered independently, the trained chaser shadows the explorer — the trajectories braid together, collision events (red bands) come in bursts whenever the explorer is cornered, and the explorer's escapes (pink bands) immediately trigger new approaches (orange).

### Playoffs against random opponents (reported in text)

The paper also benchmarks each trained agent against an opponent that acts uniformly at random, scoring the first 100 steps of each episode (their Fig. 5i–n). We verified our pair on the same protocol (100 episodes per pairing): against a random explorer, the social chaser collides **9.1** times per 100 steps, keeps the partner in vision **66%** of the time, and stays at mean distance **3.3** — the non-social chaser manages only 3.5 collisions, 33% vision, and distance 5.2. Against a random chaser, the social explorer keeps a larger mean distance (**5.3** vs 5.1 grid units) and visits fewer new fields (**51** vs 63 per 100 steps), i.e. it trades exploration for flight. All six metrics go in the paper's direction — chasing and fleeing are learned *social* behaviors, not byproducts of exploration — but with weaker magnitudes than the paper's numbers, which pool 10 agent pairs trained for 20,000 iterations (we trained one pair for 8,000). Because it adds no analysis beyond this benchmark, we report the control here in text only and move on to the behavioral signature the paper analyzes in depth: where the chaser moves relative to the explorer.

The evaluation and analysis machinery for the rest of the notebook — rollout collection with optional null-space perturbation, behavioral summaries, flow fields, SVM decoding, PLSC shared/unique subspaces, partner representation, and perturbation directions:

In [ ]:
from pathlib import Path

import numpy as np
import torch



def _events_to_bv(events: set, is_agent1: bool) -> np.ndarray:
    """One-hot event row, column order EVENT_COLUMNS; the explorer row encodes
    escapes, the chaser row encodes approach (reference elif-chain priority)."""
    bv = np.zeros(len(EVENT_COLUMNS), dtype=np.int64)
    if "collision" in events:
        bv[2] = 1
    elif ("agent1_new_field" if is_agent1 else "agent2_new_field") in events:
        bv[1] = 1
    elif is_agent1 and "escape_far" in events:
        bv[4] = 1
    elif is_agent1 and "escape_near" in events:
        bv[5] = 1
    elif is_agent1 and "escape_close" in events:
        bv[6] = 1
    elif not is_agent1 and "approach" in events:
        bv[3] = 1
    else:
        bv[0] = 1
    return bv


@torch.no_grad()
def collect_rollouts(agent1, agent2, condition: str = "social",
                     n_episodes: int = 50, episode_len: int = 500,
                     deterministic: bool = False, seed: int = 0,
                     device: str = "cpu", project_out1: np.ndarray | None = None,
                     project_out2: np.ndarray | None = None,
                     random_agent: int | None = None) -> dict:
    """Roll out the agent pair and record everything needed for the analyses.

    Args:
        agent1, agent2: framework ActorCriticModels (explorer, chaser).
        condition: arena condition ("social", "nonsocial", "nonsocial_fov7",
            "nonsocial_fullvision").
        n_episodes, episode_len: eval uses 50 x 500 in the reference.
        deterministic: argmax actions instead of sampling.
        project_out1/project_out2: optional (256, k) orthonormal bases; the
            corresponding agent's hidden state is projected onto the
            null space of these directions before the actor/critic heads.
        random_agent: playoff mode (paper Fig. 5i-n) — 1 or 2 makes the arena
            overwrite that agent's action with a uniform random one. The
            recorded A1/A2 for that agent are then the policy's own actions,
            not the executed ones; use only behavior metrics from such
            rollouts.

    Returns:
        dict of stacked arrays: R1/R2 (E,T,256), Pos_R1/Pos_R2 (E,T,2),
        Distance (E,T), Bv1/Bv2 (E,T,7), A1/A2 (E,T,4), rewards
        rew1/rew2 (E,T), partner_visible (E,T), and the weight matrices
        W_inp1/2 (200,256), W_rec1/2 (256,256), W_out1/2 (256,4).
    """
    dev = torch.device(device)
    agent1 = agent1.to(dev).eval()
    agent2 = agent2.to(dev).eval()
    P1 = None if project_out1 is None else torch.as_tensor(
        project_out1, dtype=torch.float32, device=dev)
    P2 = None if project_out2 is None else torch.as_tensor(
        project_out2, dtype=torch.float32, device=dev)

    env = make_arena(condition, episode_len=episode_len,
                     random_agent=random_agent, seed=seed)
    M = agent1.config.latent_dim

    R1 = np.zeros((n_episodes, episode_len, M), dtype=np.float32)
    R2 = np.zeros_like(R1)
    Pos1 = np.zeros((n_episodes, episode_len, 2), dtype=np.float32)
    Pos2 = np.zeros_like(Pos1)
    Dist = np.zeros((n_episodes, episode_len), dtype=np.float32)
    Bv1 = np.zeros((n_episodes, episode_len, len(EVENT_COLUMNS)), dtype=np.int64)
    Bv2 = np.zeros_like(Bv1)
    A1 = np.zeros((n_episodes, episode_len, 4), dtype=np.int64)
    A2 = np.zeros_like(A1)
    rew1 = np.zeros((n_episodes, episode_len), dtype=np.float32)
    rew2 = np.zeros_like(rew1)
    visible = np.zeros((n_episodes, episode_len), dtype=np.int64)

    for e in range(n_episodes):
        obs = env.reset()
        z1 = agent1.init_state(1, dev)
        z2 = agent2.init_state(1, dev)
        for t in range(episode_len):
            x1 = torch.as_tensor(obs["agent1"], dtype=torch.float32,
                                 device=dev).unsqueeze(0)
            x2 = torch.as_tensor(obs["agent2"], dtype=torch.float32,
                                 device=dev).unsqueeze(0)
            z1 = agent1.core.recurrence(x1, z1)
            z2 = agent2.core.recurrence(x2, z2)
            h1 = z1 if P1 is None else z1 - (z1 @ P1) @ P1.T
            h2 = z2 if P2 is None else z2 - (z2 @ P2) @ P2.T
            logits1 = agent1.actor(h1) * agent1.config.policy_logit_scale
            logits2 = agent2.actor(h2) * agent2.config.policy_logit_scale
            if deterministic:
                a1 = int(logits1.argmax(-1).item())
                a2 = int(logits2.argmax(-1).item())
            else:
                a1 = int(torch.distributions.Categorical(logits=logits1)
                         .sample().item())
                a2 = int(torch.distributions.Categorical(logits=logits2)
                         .sample().item())

            obs, rew, done, info = env.step({"agent1": a1, "agent2": a2})
            R1[e, t] = z1[0].cpu().numpy()
            R2[e, t] = z2[0].cpu().numpy()
            Pos1[e, t] = info["agent1_pos"]
            Pos2[e, t] = info["agent2_pos"]
            Dist[e, t] = float(np.linalg.norm(np.array(env.agent1_pos)
                                              - np.array(env.agent2_pos)))
            ev = info["events"]
            Bv1[e, t] = _events_to_bv(ev, is_agent1=True)
            Bv2[e, t] = _events_to_bv(ev, is_agent1=False)
            A1[e, t, a1] = 1
            A2[e, t, a2] = 1
            rew1[e, t] = rew["agent1"]
            rew2[e, t] = rew["agent2"]
            visible[e, t] = int(info["partner_visible"])

    def weights(agent):
        return {
            "W_inp": agent.core.input2h.weight.detach().cpu().numpy().T,
            "W_rec": agent.core.h2h.weight.detach().cpu().numpy().T,
            "W_out": agent.actor.weight.detach().cpu().numpy().T,
        }

    w1, w2 = weights(agent1), weights(agent2)
    return {
        "R1": R1, "R2": R2, "Pos_R1": Pos1, "Pos_R2": Pos2, "Distance": Dist,
        "Bv1": Bv1, "Bv2": Bv2, "A1": A1, "A2": A2,
        "rew1": rew1, "rew2": rew2, "partner_visible": visible,
        "W_inp1": w1["W_inp"], "W_rec1": w1["W_rec"], "W_out1": w1["W_out"],
        "W_inp2": w2["W_inp"], "W_rec2": w2["W_rec"], "W_out2": w2["W_out"],
        "condition": condition, "episode_len": episode_len,
    }


def save_rollouts(data: dict, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **data)


def load_rollouts(path: str | Path) -> dict:
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

In [ ]:
import numpy as np

# --------------------------------------------------------------------------
# Behavioral analyses (paper Fig. 5)
# --------------------------------------------------------------------------

def playoff_summary(data: dict) -> dict:
    """Behavioral metrics of one evaluation run (per episode, then averaged).

    Returns collisions, partner-in-vision fraction, mean inter-agent distance,
    and new fields visited, for explorer and chaser.
    """
    n_ep = data["Bv1"].shape[0]
    T = data["Bv1"].shape[1]
    return {
        "n_episodes": n_ep,
        "collisions": data["Bv1"][..., 2].sum(axis=1),
        "partner_in_vision": data["partner_visible"].mean(axis=1),
        "mean_distance": data["Distance"].mean(axis=1),
        "explorer_new_fields": data["Bv1"][..., 1].sum(axis=1),
        "chaser_new_fields": data["Bv2"][..., 1].sum(axis=1),
        "explorer_reward": data["rew1"].sum(axis=1),
        "chaser_reward": data["rew2"].sum(axis=1),
    }


# the 8 symmetries of the square grid (rotations + reflections), as 2x2
# matrices acting on (drow, dcol) vectors, with their determinant. The arena
# (square grid, isotropic rewards, uniform random restarts) is invariant under
# all of them, so samples can be dihedrally augmented — the single-pair analog
# of the paper's averaging the flow-field matrix across 10 agent pairs.
_D4 = [(np.array([[1, 0], [0, 1]]), +1), (np.array([[-1, 0], [0, -1]]), +1),
       (np.array([[0, 1], [-1, 0]]), +1), (np.array([[0, -1], [1, 0]]), +1),
       (np.array([[-1, 0], [0, 1]]), -1), (np.array([[1, 0], [0, -1]]), -1),
       (np.array([[0, 1], [1, 0]]), -1), (np.array([[0, -1], [-1, 0]]), -1)]


def action_flow_field(data: dict, agent: int = 2, fov: int = 3,
                      symmetrize: bool = True) -> dict:
    """Mean action vector and action-partner angle as a function of the
    partner's relative position (paper: "angle between the agent's action and
    the vector pointing from the agent to the opponent ... only when the
    opponent was in the agent's field of vision").

    Alignment: rollout arrays record *post-step* positions, so the action at
    step t was chosen from the observation generated by the positions (and
    partner visibility) of step t-1. We therefore pair ``A[:, 1:]`` with
    ``Pos[:, :-1]`` / ``partner_visible[:, :-1]``.

    Symmetrization: individual trained pairs are idiosyncratically anisotropic
    (they develop preferred approach directions); the paper averages the angle
    matrix over 10 independently trained pairs. With a single pair we instead
    average each sample over the 8 dihedral symmetries of the arena — the
    group average the 10-pair average converges to.

    Args:
        data: rollout dict.
        agent: 1 (explorer) or 2 (chaser).
        fov: field-of-view half-width (3 for the 7x7 vision).
        symmetrize: apply the 8-fold dihedral augmentation.

    Returns:
        dict with (2*fov+1)^2 matrices ``mean_dx``/``mean_dy`` (mean action
        vector in plot coords: +x = right, +y = up), ``mean_angle`` (signed
        angle between action and the to-partner direction, radians),
        ``count``, indexed [dr, dc] with dr = partner_row - self_row,
        dc = partner_col - self_col.
    """
    pos_self = data[f"Pos_R{agent}"]
    pos_other = data[f"Pos_R{2 if agent == 1 else 1}"]
    actions = data[f"A{agent}"]              # (E, T, 4) one-hot
    visible = data["partner_visible"].astype(bool)

    # action -> (drow, dcol); 0=up,1=right,2=down,3=left
    move = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]], dtype=float)
    a_vec = actions[:, 1:] @ move             # (E, T-1, 2), action at step t

    rel = pos_other[:, :-1] - pos_self[:, :-1]  # partner - self at decision time
    in_fov = visible[:, :-1] & (np.abs(rel[..., 0]) <= fov) \
        & (np.abs(rel[..., 1]) <= fov) & (np.abs(rel).sum(axis=-1) > 0)

    n = 2 * fov + 1
    count = np.zeros((n, n))
    sum_vec = np.zeros((n, n, 2))
    sum_angle = np.zeros((n, n))
    rels = rel[in_fov]                        # (N, 2)
    vecs = a_vec[in_fov]                      # (N, 2)
    norms = np.linalg.norm(rels, axis=1)
    to_p = rels / norms[:, None]
    dots = (vecs * to_p).sum(1)
    crosses = to_p[:, 0] * vecs[:, 1] - to_p[:, 1] * vecs[:, 0]
    angles = np.arctan2(crosses, dots)        # signed: to-partner -> action

    transforms = _D4 if symmetrize else _D4[:1]
    sample_angles = []
    for mat, det in transforms:
        rg = rels @ mat.T
        vg = vecs @ mat.T
        ag = det * angles                     # reflections flip the sign
        sample_angles.append(ag)
        dri = rg[:, 0].astype(int) + fov
        dci = rg[:, 1].astype(int) + fov
        np.add.at(count, (dri, dci), 1)
        np.add.at(sum_vec, (dri, dci), vg)
        np.add.at(sum_angle, (dri, dci), ag)

    with np.errstate(invalid="ignore", divide="ignore"):
        mean_vec = sum_vec / count[..., None]
        mean_angle = sum_angle / count
    return {"mean_dx": mean_vec[..., 1], "mean_dy": -mean_vec[..., 0],
            "mean_angle": mean_angle, "count": count, "fov": fov,
            "sample_angles": np.concatenate(sample_angles)}


# --------------------------------------------------------------------------
# SVM decoding of social events (paper Fig. 6b-e)
# --------------------------------------------------------------------------

def svm_decode(R: np.ndarray, labels: np.ndarray, mask: np.ndarray | None = None,
               n_shuffle: int = 100, seed: int = 0, cv: int = 5) -> dict:
    """Decode a binary event from RNN activity with a linear SVM.

    Args:
        R: (E, T, M) hidden activity.
        labels: (E, T) binary event indicator.
        mask: optional (E, T) bool; only these time steps are used (e.g.
            partner-in-vision steps for approach decoding).
        n_shuffle: label-shuffle controls.
        cv: cross-validation folds.

    Returns:
        dict with balanced accuracy ``score``, per-fold scores, shuffle
        distribution, and the number of samples/positives used.
    """
    from sklearn.svm import LinearSVC
    from sklearn.model_selection import cross_val_score

    X = R.reshape(-1, R.shape[-1])
    y = labels.reshape(-1).astype(int)
    if mask is not None:
        m = mask.reshape(-1).astype(bool)
        X, y = X[m], y[m]
    keep = (y == 1) | (y == 0)
    X, y = X[keep], y[keep]
    rng = np.random.RandomState(seed)
    if y.sum() == 0 or y.sum() == len(y):
        return {"score": np.nan, "shuffle": np.full(n_shuffle, np.nan),
                "n_samples": len(y), "n_positive": int(y.sum())}

    clf = LinearSVC(dual="auto", max_iter=5000)
    scores = cross_val_score(clf, X, y, cv=cv, scoring="balanced_accuracy")
    shuf = np.zeros(n_shuffle)
    for s in range(n_shuffle):
        ys = rng.permutation(y)
        shuf[s] = cross_val_score(clf, X, ys, cv=cv,
                                  scoring="balanced_accuracy").mean()
    return {"score": float(scores.mean()), "folds": scores,
            "shuffle": shuf, "shuffle_mean": float(shuf.mean()),
            "n_samples": len(y), "n_positive": int(y.sum())}


# --------------------------------------------------------------------------
# PLSC shared / unique subspaces (paper Fig. 6f-h; Methods "Analysis of
# shared neural dynamics using PLSC")
# --------------------------------------------------------------------------

def _zscore(X: np.ndarray) -> np.ndarray:
    return (X - X.mean(0)) / (X.std(0) + 1e-12)


def _plsc_svd(X: np.ndarray, Y: np.ndarray):
    """SVD of the cross-covariance matrix of z-scored X (T,n1), Y (T,n2).
    Returns W_X, singular values, W_Y, and per-dimension correlations."""
    Xz, Yz = _zscore(X), _zscore(Y)
    C = Xz.T @ Yz
    Wx, s, Wyt = np.linalg.svd(C, full_matrices=False)
    Wy = Wyt.T
    px = Xz @ Wx
    py = Yz @ Wy
    corr = np.array([np.corrcoef(px[:, i], py[:, i])[0, 1]
                     for i in range(px.shape[1])])
    return Wx, s, Wy, corr


def plsc_shared_subspace(X1: np.ndarray, X2: np.ndarray, n_perm: int = 10000,
                         seed: int = 0, chunk: int = 500) -> dict:
    """Identify shared dimensions between two agents' activity matrices.

    X1, X2: (T, M) hidden activity (episodes concatenated along time). The
    significance of each PLSC dimension is tested against null distributions
    of its singular value AND its cross-correlation, generated by circularly
    time-shifting one agent's activity by ``n_perm`` random lags (the paper's
    temporal-permutation test). A dimension counts as shared only if it
    exceeds the 97.5th percentile of both nulls, and only the top
    sequentially significant dimensions are counted.
    """
    rng = np.random.RandomState(seed)
    Wx, s, Wy, corr = _plsc_svd(X1, X2)
    k = s.shape[0]

    null_s = np.zeros((n_perm, k))
    null_c = np.zeros((n_perm, k))
    T = X1.shape[0]
    for p in range(n_perm):
        lag = rng.randint(1, T)
        X2s = np.roll(X2, lag, axis=0)
        _, sn, _, cn = _plsc_svd(X1, X2s)
        null_s[p] = sn
        null_c[p] = cn

    thr_s = np.percentile(null_s, 97.5, axis=0)
    thr_c = np.percentile(null_c, 97.5, axis=0)
    sig = (s > thr_s) & (corr > thr_c)
    n_shared = 0
    for v in sig:  # sequential counting
        if not v:
            break
        n_shared += 1
    return {"W_X": Wx, "W_Y": Wy, "svals": s, "corr": corr,
            "null_s": null_s, "null_c": null_c,
            "thr_s": thr_s, "thr_c": thr_c, "significant": sig,
            "n_shared": n_shared,
            "PLSC1_corr": corr[0], "dPLSC1_corr": corr[0] - null_c[:, 0].mean()}


def unique_subspace(X: np.ndarray, W_shared: np.ndarray) -> dict:
    """PCA of the activity projected onto the null space of the shared
    dimensions (paper: "we projected the neural activity ... onto this
    subspace and performed PCA to establish the unique neural subspace")."""
    Xz = _zscore(X)
    if W_shared.size:
        Xr = Xz - (Xz @ W_shared) @ W_shared.T
    else:
        Xr = Xz
    cov = Xr.T @ Xr / Xr.shape[0]
    evals, evecs = np.linalg.eigh(cov)
    order = np.argsort(evals)[::-1]
    return {"W": evecs[:, order], "var": evals[order]}


def tail_subspace_pc(X: np.ndarray, W: np.ndarray, corr: np.ndarray,
                     corr_max: float = 0.05) -> np.ndarray:
    """Top PC of the activity within the least-coupled PLSC dimensions,
    mapped back to unit space.

    Fallback for the paper's unique-subspace construction when *every* PLSC
    dimension passes the significance test (so the non-significant set is
    empty): we take the dimensions whose cross-agent correlation is below
    ``corr_max`` (i.e. at the chance level of the circular-shift null),
    project the z-scored activity onto their span, and return the first PC
    expressed in the original 256-d unit space.
    """
    Xz = _zscore(X)
    tail = np.where(corr < corr_max)[0]
    if len(tail) == 0:
        tail = np.argsort(corr)[:8]
    Wt = W[:, tail]
    proj = Xz @ Wt                        # (T, k) activity in the tail span
    cov = proj.T @ proj / proj.shape[0]
    evals, evecs = np.linalg.eigh(cov)
    return Wt @ evecs[:, np.argmax(evals)]


def significant_unit_groups(w_a: np.ndarray, w_b: np.ndarray,
                            thresh_sd: float = 1.5) -> dict:
    """The paper's significant-unit analysis (Fig. 6i,j; Methods): a unit
    contributes significantly to a dimension if its absolute weight exceeds
    the mean + ``thresh_sd`` s.d. of that dimension's absolute-weight
    distribution. Returns the boolean masks for the units significant only
    for dimension a, only for b, or for both, plus the per-unit difference
    |w_a| - |w_b| that the paper histograms."""
    wa, wb = np.abs(w_a), np.abs(w_b)
    sa = wa > wa.mean() + thresh_sd * wa.std()
    sb = wb > wb.mean() + thresh_sd * wb.std()
    return {"only_a": sa & ~sb, "only_b": sb & ~sa, "both": sa & sb,
            "diff": wa - wb}


def plsc1_lag_profile(X1: np.ndarray, X2: np.ndarray, Wx1: np.ndarray,
                      Wy1: np.ndarray, max_lag: int = 50) -> dict:
    """Cross-correlation of the top shared dimension across temporal lags
    (paper's temporal-lag analysis, Extended Data)."""
    x = _zscore(X1) @ Wx1
    y = _zscore(X2) @ Wy1
    lags = np.arange(-max_lag, max_lag + 1)
    cc = np.array([np.corrcoef(x, np.roll(y, l))[0, 1] for l in lags])
    return {"lags": lags, "corr": cc}


def plsc1_disruption(X1: np.ndarray, X2: np.ndarray, Wx1: np.ndarray,
                     Wy1: np.ndarray, n_perm: int = 100,
                     seed: int = 0) -> dict:
    """Cross-covariance of PLSC1 after disrupting intra-network correlation or
    temporal coupling (paper Fig. 6k), each with its shuffle baseline
    subtracted.

    - original: cov(X1 w1, X2 w2) minus the value after circular time shifts.
    - remove intra-network correlation: independently permute the units of
      each agent in time (each unit gets its own random circular shift), which
      destroys between-unit correlation structure while preserving each
      unit's own temporal structure, then recompute the PLSC1 covariance.
    - remove temporal coupling: circularly shift one agent's whole activity
      matrix by a random lag.
    """
    rng = np.random.RandomState(seed)
    x = _zscore(X1) @ Wx1
    y = _zscore(X2) @ Wy1
    T = X1.shape[0]

    def cov_shifted(a, b):
        return float(np.mean([np.cov(a, np.roll(b, rng.randint(1, T)))[0, 1]
                              for _ in range(n_perm)]))

    cov_orig = float(np.cov(x, y)[0, 1])
    out = {"original": cov_orig - cov_shifted(x, y)}

    # disrupt intra-network correlation: per-unit independent time shifts
    def perunit_shift(X):
        return np.stack([np.roll(X[:, u], rng.randint(1, T))
                         for u in range(X.shape[1])], axis=1)

    cov_intra = []
    for _ in range(n_perm):
        X1p, X2p = perunit_shift(X1), perunit_shift(X2)
        cov_intra.append(float(np.cov(_zscore(X1p) @ Wx1,
                                      _zscore(X2p) @ Wy1)[0, 1]))
    out["remove_intra"] = float(np.mean(cov_intra)) - cov_shifted(x, y)

    # disrupt temporal coupling: shift one agent as a whole
    out["remove_temporal"] = float(np.mean([
        np.cov(x, np.roll(y, rng.randint(1, T)))[0, 1]
        for _ in range(n_perm)])) - cov_shifted(x, y)
    return out


def concat_activity(data: dict, agent: int = 1, exclude_degenerate: bool = True,
                    stick_threshold: float = 0.01) -> tuple[np.ndarray, np.ndarray]:
    """Concatenate episodes' hidden activity (and mask), excluding episodes
    where the agent shows degenerate behavior: the paper excludes "episodes
    where agents ceased moving in the same state or repeated movements within
    the same tile for more than 1% of the episode length". We implement this
    as a *consecutive* run of steps on the same tile longer than 1% of the
    episode (the total fraction would exclude every good chase, since a
    blocked collision attempt also leaves the position unchanged)."""
    R = data[f"R{agent}"]
    keep = np.ones(R.shape[0], dtype=bool)
    if exclude_degenerate:
        T = R.shape[1]
        max_stick = int(np.ceil(stick_threshold * T))
        # exclude the episode if EITHER agent is degenerate (the paper's
        # "episodes where agents ceased moving"), so the two agents' episode
        # sets stay aligned for pairwise analyses like PLSC.
        for pos in (data["Pos_R1"], data["Pos_R2"]):
            for e in range(R.shape[0]):
                if not keep[e]:
                    continue
                same_tile = np.all(np.diff(pos[e], axis=0) == 0, axis=1)
                run = 0
                for v in same_tile:
                    run = run + 1 if v else 0
                    if run > max_stick:
                        keep[e] = False
                        break
    return R[keep].reshape(-1, R.shape[-1]), keep


# --------------------------------------------------------------------------
# Partner-behavior representation (paper Fig. 6l,m; Methods "Calculating
# non-redundant neural variance explained")
# --------------------------------------------------------------------------

def behavior_predictors(data: dict, agent: int = 1,
                        keep: np.ndarray | None = None) -> dict[str, np.ndarray]:
    """Build the predictor matrix: collision, self/partner x,y positions,
    self/partner action one-hots, self/partner behaviors (new field;
    approach for the chaser; escape for the explorer).

    ``keep``: optional boolean per-episode mask (from ``concat_activity``)
    to drop degenerate episodes, keeping alignment with the activity matrix.
    """
    E, T = data["Bv1"].shape[:2]
    other = 2 if agent == 1 else 1
    bv_self, bv_other = data[f"Bv{agent}"], data[f"Bv{other}"]
    pos_self = data[f"Pos_R{agent}"] / 9.0          # normalized to [0, 1]
    pos_other = data[f"Pos_R{other}"] / 9.0
    a_self, a_other = data[f"A{agent}"], data[f"A{other}"]
    if keep is not None:
        bv_self, bv_other = bv_self[keep], bv_other[keep]
        pos_self, pos_other = pos_self[keep], pos_other[keep]
        a_self, a_other = a_self[keep], a_other[keep]
        E = int(keep.sum())
    collision = bv_self[..., 2:3]
    new_self, new_other = bv_self[..., 1:2], bv_other[..., 1:2]
    if agent == 2:   # chaser: approach; explorer behavior for partner
        beh_self = bv_self[..., 3:4]
        beh_other = (bv_other[..., 4:7].sum(-1, keepdims=True) > 0).astype(float)
    else:            # explorer: escape (any); approach for partner
        beh_self = (bv_self[..., 4:7].sum(-1, keepdims=True) > 0).astype(float)
        beh_other = bv_other[..., 3:4]
    parts = {
        "collision": collision.reshape(E * T, 1),
        "self_pos": pos_self.reshape(E * T, 2),
        "partner_pos": pos_other.reshape(E * T, 2),
        "self_action": a_self.reshape(E * T, 4).astype(float),
        "partner_action": a_other.reshape(E * T, 4).astype(float),
        "self_behavior": new_self.reshape(E * T, 1),
        "partner_behavior": new_other.reshape(E * T, 1),
        "self_event": beh_self.reshape(E * T, 1),
        "partner_event": beh_other.reshape(E * T, 1),
    }
    return parts


def _fit_r2(X: np.ndarray, Y: np.ndarray) -> float:
    """Multivariate OLS fraction of variance of Y explained by X (pooled over
    target dims, intercept included)."""
    X1 = np.concatenate([X, np.ones((X.shape[0], 1))], axis=1)
    beta, *_ = np.linalg.lstsq(X1, Y, rcond=None)
    pred = X1 @ beta
    ss_res = ((Y - pred) ** 2).sum()
    ss_tot = ((Y - Y.mean(0)) ** 2).sum() + 1e-12
    return float(1 - ss_res / ss_tot)


def partner_representation(R: np.ndarray, predictors: dict[str, np.ndarray],
                           n_perm: int = 100, seed: int = 0) -> dict:
    """Fraction of the *neural* variance explained by the behavior predictors
    (paper Fig. 6l,m,p; Methods "Calculating non-redundant neural variance
    explained"): each neural unit's activity is regressed on the predictor
    matrix.

    R: (T, M) neural activity (full hidden space, a PLSC dimension, or the
    neural action space ``W_action W_action.T @ h``).

    Returns chance-subtracted variance explained by: all predictors
    (``r2_all``), partner predictors alone (``r2_partner``), partner
    predictors beyond what self predictors explain (``r2_partner_unique``),
    and the permutation-based non-redundant partner contribution
    (``partner_contribution``: drop in explained variance when the partner
    predictors are circularly shifted in time). Chance levels from circular
    shifts of the predictor matrix (100 permutations in the paper).
    """
    rng = np.random.RandomState(seed)
    T = R.shape[0]
    names = list(predictors)
    F = np.concatenate([predictors[k] for k in names], axis=1)
    is_partner = np.concatenate(
        [np.full(predictors[k].shape[1], k.startswith("partner"))
         for k in names]).astype(bool)
    F_partner, F_self = F[:, is_partner], F[:, ~is_partner]

    def chance_sub(Fx):
        r2 = _fit_r2(Fx, R)
        r2_null = np.mean([_fit_r2(np.roll(Fx, rng.randint(1, T), axis=0), R)
                           for _ in range(n_perm)])
        return r2 - r2_null

    r2_all = chance_sub(F)
    r2_partner = chance_sub(F_partner)
    r2_self = chance_sub(F_self)
    # non-redundant partner contribution: permute partner predictors together
    r2_perm = np.mean([
        _fit_r2(np.where(is_partner[None, :],
                         np.roll(F, rng.randint(1, T), axis=0), F), R)
        for _ in range(n_perm)])
    return {"r2_all": r2_all, "r2_partner": r2_partner,
            "r2_partner_unique": r2_all - r2_self,
            "partner_contribution": _fit_r2(F, R) - r2_perm}


def neural_action_space(R: np.ndarray, W_out: np.ndarray) -> np.ndarray:
    """Project hidden activity into the neural action space:
    h_action = W_action W_action^T h  (W_out is (256, 4) as stored in the
    rollout dict, i.e. the transposed actor weight)."""
    W = W_out.T                       # (4, 256)
    return (R @ (W.T @ W).T)          # (T, 256)


# --------------------------------------------------------------------------
# Perturbation control directions (paper Fig. 6s-u + Extended Data Fig. 13g)
# --------------------------------------------------------------------------

def perturbation_directions(R: np.ndarray, W_shared: np.ndarray,
                            n_plsc: int = 10, n_control: int = 25,
                            seed: int = 0) -> dict:
    """Directions to project out for the causal perturbation.

    - plsc: the top ``n_plsc`` shared dimensions (columns of W_shared).
    - control: ``n_control`` random principal components that do not overlap
      the top PLSCs — PCA on the temporally permuted, null-space-projected
      activity (paper Methods; they account for about the same variance as
      the top 10 PLSCs).
    """
    rng = np.random.RandomState(seed)
    T, M = R.shape
    P = W_shared[:, :n_plsc]
    # project into null space of the PLSCs, then temporally permute each unit
    Xn = R - (R @ P) @ P.T
    Xp = np.stack([np.roll(Xn[:, u], rng.randint(1, T)) for u in range(M)],
                  axis=1)
    cov = Xp.T @ Xp / T
    evals, evecs = np.linalg.eigh(cov)
    order = np.argsort(evals)[::-1]
    Q = evecs[:, order[:n_control]]
    var_plsc = float(((R @ P) ** 2).sum() / (R ** 2).sum())
    var_ctrl = float(((R @ Q) ** 2).sum() / (R ** 2).sum())
    return {"plsc": P, "control": Q,
            "var_plsc": var_plsc, "var_control": var_ctrl}

### Movement flow fields

A more direct look at the learned policy: whenever the chaser sees the explorer, how does its movement direction relate to the explorer's direction? For every step with the explorer inside the chaser's field of vision we compute the signed angle between the chaser's action and the straight line to the explorer, and bin actions by the explorer's relative position (the paper's Fig. 5o–r, computed at an early and a late checkpoint of the social run).

Two technical points. First, *alignment*: the rollout arrays record positions **after** each step, while the action was chosen from the observation generated by the **previous** step's positions — so we pair action $t$ with positions $t-1$. Second, *averaging*: the paper averages the angle matrix over 10 independently trained pairs, which washes out the idiosyncratic directional preferences any single pair develops (any single pair, ours included, is visibly anisotropic on its own). Since the arena is fully symmetric under the 8 rotations/reflections of the square grid — isotropic rewards, symmetric walls, uniform random restarts — we average each sample over those 8 symmetries instead, the single-pair analog of the paper's 10-pair average.

In [ ]:
def flow_field_from_checkpoints(condition, names=("checkpoint-iter200", "final"),
                                n_episodes=25, seed=200):
    """Flow fields at two training stages (early / late)."""
    labels = {"checkpoint-iter200": "early in training",
              "final": "late in training"}
    out = {}
    for name in names:
        cache = ANALYSIS_DIR / f"flow_{condition}_{name}.npz"
        if cache.exists():
            d = load_rollouts(cache)
        else:
            d0 = PAIR_DIRS[condition] / name
            if not (d0 / "agent1").exists():
                print(f"[{condition}] {name} not available")
                continue
            a1 = AutoModel.from_pretrained(d0 / "agent1")
            a2 = AutoModel.from_pretrained(d0 / "agent2")
            d = collect_rollouts(a1, a2, condition=condition,
                                 n_episodes=n_episodes, episode_len=500,
                                 seed=seed, device=DEVICE)
            save_rollouts(d, cache)
        out[labels.get(name, name)] = action_flow_field(d, agent=2)
    return out


def plot_flow_fields(ffs, min_count=80):
    """Quiver flow fields + polar + bar angle distributions (Fig. 5o-r)."""
    names = list(ffs)
    fig = plt.figure(figsize=(12.5, 3.3))
    axes = [fig.add_subplot(1, 4, i + 1) for i in range(2)]
    axp = fig.add_subplot(1, 4, 3, projection="polar")
    axb = fig.add_subplot(1, 4, 4)
    stage_colors = {"early in training": "gray", "late in training": "#f5a623"}

    for ax, name in zip(axes, names):
        ff = ffs[name]
        fov = ff["fov"]
        DR, DC = np.mgrid[-fov:fov + 1, -fov:fov + 1]  # bin (drow, dcol)
        # draw each arrow at the CHASER's position relative to the explorer
        # (the explorer sits at the origin), so a converging field points inward
        m = ff["count"] >= min_count
        ax.quiver(-DC[m], DR[m], ff["mean_dx"][m], ff["mean_dy"][m],
                  scale=5, width=0.010, color="#d62728")
        ax.plot(0, 0, "o", color="#1f77b4", ms=8, zorder=5)  # the explorer
        ax.set_xlim(-fov - 0.7, fov + 0.7); ax.set_ylim(-fov - 0.7, fov + 0.7)
        ax.set_aspect("equal")
        ax.set_xticks(range(-fov, fov + 1)); ax.set_yticks(range(-fov, fov + 1))
        ax.set_xlabel("chaser's relative x")
        ax.set_ylabel("chaser's relative y")
        ax.set_title(name, fontsize=9)

    bins = np.linspace(-np.pi, np.pi, 13)  # 12 bins of 30 deg
    for name in names:
        ang = ffs[name]["sample_angles"]
        hist, edges = np.histogram(ang, bins=bins)
        hist = hist / hist.sum()
        theta = 0.5 * (edges[1:] + edges[:-1])
        color = stage_colors.get(name, "k")
        axp.plot(theta, hist, color=color, lw=1.8, label=name)
        axp.fill_between(theta, hist, color=color, alpha=0.2)
        axb.bar(np.degrees(theta) + (-3 if "early" in name else 3), hist,
                width=6, color=color, alpha=0.8, label=name)
        frac60 = (np.abs(ang) <= np.pi / 3).mean()
        print(f"{name}: {(ang.size)} in-vision steps (8-fold symmetrized), "
              f"{frac60:.0%} within +/-60 deg of the line to the explorer")
    axp.set_theta_zero_location("N")
    axp.set_title("action angle vs direction\nto explorer (polar)", pad=14)
    axb.set_xlabel("angle (degrees)")
    axb.set_ylabel("probability")
    axb.set_title("angle distribution (bar)")
    axb.legend(fontsize=8)
    fig.tight_layout()
    return fig


flows = flow_field_from_checkpoints("social")
if len(flows) == 2:
    fig = plot_flow_fields(flows)
    savefig(fig, "fig_flow_fields.png")
    plt.show()

Early in training the flow field is short and disorganized — the chaser's moves barely depend on where the explorer is, and the angle distribution is close to uniform. Late in training every arrow points at the explorer: from wherever the chaser stands, its mean action heads straight for the blue dot, and the angle distribution piles up around $0°$ (a chance-level mover would put ~33% of steps within $\pm 60°$; our late-stage chaser is near 60%). The residual $\pm 45°$ shoulders are a grid artifact — when the explorer is on a diagonal, the best axis-aligned move is half a cell off the direct line.

## Neural dynamics

We now open the networks. The evaluation protocol follows the paper: 25 episodes of 500 steps per condition, dropping episodes in which either agent ceases moving for more than 1% of the episode (degenerate behavior), then concatenating the remaining episodes in time. All neural analyses below work on the hidden activity $h_t \in \mathbb{R}^{256}$ of both agents recorded during these rollouts.

In [ ]:
def eval_rollouts(condition, n_episodes=25, episode_len=500, seed=0):
    """Main evaluation rollouts for a condition (cached)."""
    cache = ANALYSIS_DIR / f"eval_{condition}.npz"
    if cache.exists():
        return load_rollouts(cache)
    if pairs[condition] is None:
        return None
    d = collect_rollouts(*pairs[condition], condition=condition,
                         n_episodes=n_episodes, episode_len=episode_len,
                         seed=seed, device=DEVICE)
    save_rollouts(d, cache)
    return d


evals = {}
for cond in ("social", "nonsocial"):
    d = eval_rollouts(cond)
    if d is not None:
        evals[cond] = d
        _, keep = concat_activity(d, agent=1)
        print(f"[{cond}] {keep.sum()}/{len(keep)} episodes kept after "
              f"degenerate-episode exclusion")

### Behavior and activity, side by side

The paper's Fig. 6a shows a raster of behavior and individual units' activity in both agents. Here is the same view for one episode of our social pair: event bands on top, five example hidden units per agent below.

In [ ]:
def plot_raster(data, episode=0, n_units=5, seed=1):
    """Fig. 6a analog: event bands + example unit traces for both agents."""
    rng = np.random.RandomState(seed)
    T = data["Bv1"].shape[1]
    fig, axes = plt.subplots(3, 1, figsize=(8.5, 4.6), sharex=True,
                             gridspec_kw={"height_ratios": [1, 2, 2],
                                          "hspace": 0.12})
    axb, ax1, ax2 = axes
    for t in range(T):
        bv1, bv2 = data["Bv1"][episode, t], data["Bv2"][episode, t]
        c1 = ["none", "new_field", "collision", "approach", "escape_far",
              "escape_near", "escape_close"][int(bv1.argmax())]
        c2 = ["none", "new_field", "collision", "approach", "escape_far",
              "escape_near", "escape_close"][int(bv2.argmax())]
        axb.axvspan(t, t + 1, 0.52, 1.0, color=EVENT_COLORS[c1])
        axb.axvspan(t, t + 1, 0.0, 0.48, color=EVENT_COLORS[c2])
    axb.set_xlim(0, T); axb.set_ylim(0, 1)
    axb.set_yticks([0.26, 0.76]); axb.set_yticklabels(["chaser", "explorer"],
                                                     fontsize=8)
    for ax, R, name, color in ((ax1, "R2", "chaser", "#d62728"),
                               (ax2, "R1", "explorer", "#1f77b4")):
        units = rng.choice(data[R].shape[-1], n_units, replace=False)
        for i, u in enumerate(units):
            x = data[R][episode, :, u]
            x = (x - x.min()) / (x.max() - x.min() + 1e-9)
            ax.plot(np.arange(T), x * 0.8 + i, lw=0.7, color=color)
        ax.set_yticks(range(n_units))
        ax.set_yticklabels([f"unit {u}" for u in units], fontsize=6.5)
        ax.set_ylabel(name, fontsize=9)
        ax.set_ylim(-0.4, n_units - 0.2)
    ax2.set_xlabel("time step")
    return fig


if "social" in evals:
    fig = plot_raster(evals["social"])
    savefig(fig, "fig_raster.png")
    plt.show()

**Reading the result.** Individual units are strongly modulated by the behavioral events — note how chaser units ramp when the explorer enters its FOV (approach events, orange) and how both agents' activity shifts at collisions (red). This suggests the events should be decodable from the population, which we test next.

### Decoding social events from population activity

Can the ongoing social interaction be read out from each network? We train linear SVMs to decode, from the 256-dimensional activity: **collision** and the **partner's escape** from the chaser; **collision** and the **partner's approach** from the explorer (escape/approach are decoded only on steps where the partner is inside the field of vision, as in the paper). Performance is cross-validated **balanced accuracy**, compared against label-shuffled controls.

In [ ]:
def run_decoding(data, n_shuffle=20, seed=0):
    """The four decoding problems of Fig. 6b-e, with shuffle controls."""
    vis = data["partner_visible"].astype(bool)
    escape1 = (data["Bv1"][..., 4:7].sum(-1) > 0).astype(float)
    approach2 = data["Bv2"][..., 3].astype(float)
    collision = data["Bv1"][..., 2].astype(float)
    out = {
        "chaser_collision": svm_decode(data["R2"], collision,
                                       n_shuffle=n_shuffle, seed=seed),
        "chaser_partner_escape": svm_decode(data["R2"], escape1, mask=vis,
                                            n_shuffle=n_shuffle, seed=seed),
        "explorer_collision": svm_decode(data["R1"], collision,
                                         n_shuffle=n_shuffle, seed=seed),
        "explorer_partner_approach": svm_decode(data["R1"], approach2,
                                                mask=vis, n_shuffle=n_shuffle,
                                                seed=seed),
    }
    return {k: {kk: (vv.tolist() if isinstance(vv, np.ndarray) else vv)
                for kk, vv in v.items()} for k, v in out.items()}


def get_decoding(condition, n_shuffle=20):
    cache = ANALYSIS_DIR / f"decoding_{condition}.json"
    if cache.exists():
        return json.load(open(cache))
    if condition not in evals:
        return None
    t0 = time.time()
    res = run_decoding(evals[condition], n_shuffle=n_shuffle)
    json.dump(res, open(cache, "w"), indent=1)
    print(f"[{condition}] decoding done in {time.time() - t0:.0f}s")
    return res


decodings = {c: get_decoding(c) for c in evals}


def plot_decoding(decodings):
    fig, axes = plt.subplots(1, 4, figsize=(9.5, 2.7))
    titles = ["collision\n(chaser)", "partner's escape\n(chaser)",
              "collision\n(explorer)", "partner's approach\n(explorer)"]
    keys = ["chaser_collision", "chaser_partner_escape",
            "explorer_collision", "explorer_partner_approach"]
    for ax, key, title in zip(axes, keys, titles):
        vals, labels, colors = [], [], []
        for cond, color in (("social", COLOR_SOCIAL),
                            ("nonsocial", COLOR_NONSOCIAL)):
            d = decodings.get(cond)
            if d is None:
                continue
            vals.append([d[key]["score"]])
            labels.append(cond)
            colors.append(color)
            ax.scatter(np.full(len(d[key]["shuffle"]), len(vals)),
                       d[key]["shuffle"], color="0.7", s=4, zorder=1)
        bp = ax.boxplot(vals, patch_artist=True, widths=0.55,
                        medianprops=dict(color="k"),
                        flierprops=dict(markersize=0))
        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c); patch.set_alpha(0.8)
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_ylim(0.4, 1.02)
        ax.axhline(0.5, color="k", ls=":", lw=0.8)
        ax.set_title(title, fontsize=8.5)
        ax.set_ylabel("balanced accuracy" if ax is axes[0] else "")
    fig.tight_layout()
    return fig


if any(decodings.values()):
    fig = plot_decoding(decodings)
    savefig(fig, "fig_svm_decoding.png")
    plt.show()
    for cond, d in decodings.items():
        if d:
            print(cond, {k: round(v["score"], 3) for k, v in d.items()})

In social agents, collisions and the *partner's* social behavior — escape for the chaser, approach for the explorer — are decoded far above shuffle, while in non-social agents the partner events are at chance (they are neither visible nor rewarded). The networks therefore carry a specific representation of the partner's behavior, not just of the agent's own actions. Where does that representation live? That is the question the shared-subspace analysis answers.

### The shared neural subspace (PLSC)

To find the dimensions along which the two networks' activity co-varies, we use **partial least squares correlation (PLSC)** on the z-scored activity matrices $X \in \mathbb{R}^{T\times 256}$ (explorer) and $Y \in \mathbb{R}^{T\times 256}$ (chaser) of concatenated episodes. PLSC factorizes the cross-covariance matrix

$$C = X^\top Y = W_X \Delta W_Y^\top$$

so that the projections $X w_{X,i}$ and $Y w_{Y,i}$ are maximally correlated for each dimension $i$. Unlike CCA, PLSC does not whiten within each network, so it is driven by high-variance directions that couple across agents — the regime in which inter-agent coupling should appear.

**Significance.** A dimension counts as *shared* only if both its singular value and its cross-agent correlation exceed the 97.5th percentile of null distributions generated by circularly time-shifting one agent's activity by 10,000 random lags, and only the top *sequentially* significant dimensions are counted (if dimensions 1, 2, 3 and 5 pass but 4 does not, the count is 3). The complementary **unique subspace** is obtained by projecting each agent's activity onto the null space of the shared dimensions and running PCA there. We also report $\Delta$PCC of PLSC1: the observed top-dimension correlation minus the temporally shuffled control.

In [ ]:
def get_plsc(condition, n_perm=10000, seed=0):
    """PLSC shared-subspace analysis for one condition (cached)."""
    cache = ANALYSIS_DIR / f"plsc_{condition}.npz"
    if cache.exists():
        z = np.load(cache)
        return {k: z[k] for k in z.files}
    if condition not in evals:
        return None
    R1, _ = concat_activity(evals[condition], agent=1)
    R2, _ = concat_activity(evals[condition], agent=2)
    t0 = time.time()
    res = plsc_shared_subspace(R1, R2, n_perm=n_perm, seed=seed)
    np.savez(cache, **{k: v for k, v in res.items()})
    print(f"[{condition}] PLSC done in {time.time() - t0:.0f}s: "
          f"{res['n_shared']} shared dims, PLSC1 corr {res['PLSC1_corr']:.3f}, "
          f"dPCC {res['dPLSC1_corr']:.3f}")
    return res


plscs = {c: get_plsc(c) for c in evals}

In [ ]:
def plot_plsc_summary(plscs):
    """Fig. 6f-h analog: example shared dims, counts, dPCC."""
    fig = plt.figure(figsize=(11, 3.2))
    axf = fig.add_subplot(1, 3, (1, 2))
    axg = fig.add_subplot(2, 3, 3)
    axh = fig.add_subplot(2, 3, 6)

    d = plscs.get("social")
    if d is not None and "social" in evals:
        ev = evals["social"]
        R1, keep = concat_activity(ev, agent=1)
        R2, _ = concat_activity(ev, agent=2)
        mu1, sd1 = R1.mean(0), R1.std(0) + 1e-12
        mu2, sd2 = R2.mean(0), R2.std(0) + 1e-12
        # an event-rich kept episode, windowed around its first collision
        n_coll = ev["Bv1"][..., 2].sum(1)
        cand = np.where(keep & (n_coll >= 3))[0]
        ep = int(cand[0]) if len(cand) else int(np.argmax(n_coll))
        t_coll = int(np.argmax(ev["Bv1"][ep, :, 2] > 0))
        t0 = int(np.clip(t_coll - 40, 0, ev["Bv1"].shape[1] - 100))
        sl = slice(t0, t0 + 100)
        x1 = (ev["R1"][ep] - mu1) / sd1        # z-scored with global stats
        x2 = (ev["R2"][ep] - mu2) / sd2
        for i in range(4):
            p1, p2 = x1[sl] @ d["W_X"][:, i], x2[sl] @ d["W_Y"][:, i]
            s = np.sqrt(p1.var() + p2.var()) + 1e-12
            off = 3.2 * (3 - i)
            axf.plot(np.arange(100), p1 / s + off, color="#1f77b4", lw=0.9)
            axf.plot(np.arange(100), p2 / s + off, color="#d62728", lw=0.9,
                     alpha=0.75)
            axf.text(-2, off, f"PLSC{i + 1}\n$r$={d['corr'][i]:.2f}",
                     fontsize=7, ha="right", va="center")
        # behavior bands on top: explorer (upper) and chaser (lower)
        names = ["none", "new_field", "collision", "approach", "escape_far",
                 "escape_near", "escape_close"]
        for t in range(100):
            c1 = names[int(ev["Bv1"][ep, t0 + t].argmax())]
            c2 = names[int(ev["Bv2"][ep, t0 + t].argmax())]
            axf.axvspan(t, t + 1, 1.06, 1.10, color=EVENT_COLORS[c1],
                        clip_on=False)
            axf.axvspan(t, t + 1, 1.01, 1.05, color=EVENT_COLORS[c2],
                        clip_on=False)
        from matplotlib.transforms import blended_transform_factory
        tr = blended_transform_factory(axf.transData, axf.transAxes)
        axf.text(-2, 1.08, "explorer", transform=tr, fontsize=6.5,
                 ha="right", va="center", color="0.35", clip_on=False)
        axf.text(-2, 1.03, "chaser", transform=tr, fontsize=6.5,
                 ha="right", va="center", color="0.35", clip_on=False)
        axf.set_xlim(0, 100)
        axf.set_yticks([])
        axf.set_xlabel(f"time step (episode {ep}, steps {t0}–{t0 + 100})")
        axf.set_title("shared dimensions: explorer (blue) vs chaser (red)",
                      pad=26)
        axf.spines["left"].set_visible(False)

    vals, labels, colors = [], [], []
    for cond, color, lab in (("social", COLOR_SOCIAL, "social"),
                             ("nonsocial", COLOR_NONSOCIAL, "non-social")):
        if plscs.get(cond) is not None:
            vals.append(int(plscs[cond]["n_shared"]))
            labels.append(lab); colors.append(color)
    axg.bar(range(len(vals)), vals, color=colors, width=0.6)
    axg.set_xticks(range(len(vals))); axg.set_xticklabels(labels, fontsize=8)
    axg.set_ylabel("shared dimensions")
    axg.set_title("number of shared dims", fontsize=8.5)

    vals = [float(plscs[c]["dPLSC1_corr"]) for c in
            ("social", "nonsocial") if plscs.get(c) is not None]
    axh.bar(range(len(vals)), vals, color=colors, width=0.6)
    axh.set_xticks(range(len(vals))); axh.set_xticklabels(labels, fontsize=8)
    axh.set_ylabel("$\\Delta$PCC of PLSC1")
    axh.axhline(0, color="k", ls=":", lw=0.8)
    axh.set_title("top shared dim above chance", fontsize=8.5)
    fig.tight_layout()
    return fig


if any(plscs.values()):
    fig = plot_plsc_summary(plscs)
    savefig(fig, "fig_plsc.png")
    plt.show()

The top shared dimensions track each other across the two networks ($r \approx 0.5$–$0.6$ moment by moment, far above the shuffle null of $\approx 0$), and their swings are organized by the social events shown in the bands — approaches, escapes, collisions. In our pair the coupling is so strong that **all 256 dimensions** pass the sequential permutation test (the paper reports roughly two hundred on average across its 10 pairs). Non-social training — no partner vision, no social reward — produces no significant shared dimensions at all, and its $\Delta$PCC is at zero. The shared subspace is therefore a product of *social interaction*, not of two identical architectures in identical arenas. (With the vision-control conditions `nonsocial_fov7` / `nonsocial_fullvision` trained, their bars can be added here to separate the contribution of merely *seeing* the partner — the paper finds vision alone suffices for a large shared subspace but not for elevated $\Delta$PCC.)

### Which units carry the shared signal, and what coupling does it need?

Two follow-ups from the paper. First, are the shared and unique subspaces carried by *different* units? Following the paper's Methods, we call a unit a significant contributor to a dimension if its absolute weight exceeds the mean + 1.5 s.d. of that dimension's absolute-weight distribution, and we histogram $|w_{PLSC1}| - |w_{U1}|$ separately for the units significant only for PLSC1, only for U1, or for both (and the same for PLSC1 vs PLSC2). One caveat: in our pair *all* 256 PLSC dimensions pass the significance test, so the paper's unique subspace — the span of the *non-significant* dimensions — is empty; as a stand-in for U1 we use the first PC of the activity within the least-coupled PLSC dimensions (cross-agent correlation below the shuffle level, $<0.05$), the bottom end of the same spectrum. Second, is the inter-agent correlation just a byproduct of temporally smooth activity? We recompute the PLSC1 cross-covariance after disrupting the intra-network correlation structure (independently time-shifting each unit) or the temporal coupling (shifting one agent as a whole), subtracting shuffle baselines.

In [ ]:
def plot_weights_and_disruption(condition="social", seed=0):
    """Fig. 6i-k analog: significant-unit histograms + coupling disruption."""
    d, ev = plscs[condition], evals[condition]
    R1, _ = concat_activity(ev, agent=1)
    R2, _ = concat_activity(ev, agent=2)
    n_sh = int(d["n_shared"])
    # unique direction per agent: PCA in the null space of the shared dims,
    # or — when every dim is significant (our pair) — the first PC of the
    # least-coupled PLSC dims (cross-agent corr at the shuffle level)
    u1 = {}
    for agent, R, W in ((1, R1, d["W_X"]), (2, R2, d["W_Y"])):
        if n_sh < W.shape[1]:
            u1[agent] = unique_subspace(R, W[:, :n_sh])["W"][:, 0]
        else:
            u1[agent] = tail_subspace_pc(R, W, d["corr"])
    dis = plsc1_disruption(R1, R2, d["W_X"][:, 0], d["W_Y"][:, 0],
                           n_perm=100, seed=seed)

    fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.9))
    bins = np.linspace(-0.3, 0.3, 41)
    for ax, key_b, lab_b in ((axes[0], "u1", "U1"), (axes[1], "plsc2", "PLSC2")):
        diffs = {"a": [], "b": [], "both": []}
        for agent, W in ((1, d["W_X"]), (2, d["W_Y"])):
            wb = u1[agent] if key_b == "u1" else W[:, 1]
            g = significant_unit_groups(W[:, 0], wb)
            diffs["a"].append(g["diff"][g["only_a"]])
            diffs["b"].append(g["diff"][g["only_b"]])
            diffs["both"].append(g["diff"][g["both"]])
        da = np.concatenate(diffs["a"])
        db = np.concatenate(diffs["b"])
        dbo = np.concatenate(diffs["both"])
        ax.hist(db, bins=bins, alpha=0.75, color="#1f77b4",
                label=f"{lab_b} units (n={len(db)})")
        ax.hist(da, bins=bins, alpha=0.75, color="#d62728",
                label=f"PLSC1 units (n={len(da)})")
        if len(dbo):
            ax.hist(dbo, bins=bins, alpha=0.75, color="#9467bd",
                    label=f"both (n={len(dbo)})")
        ax.axvline(0, color="k", ls=":", lw=0.8)
        ax.set_xlabel(f"$|w_{{PLSC1}}| - |w_{{{lab_b}}}|$")
        ax.set_ylabel("unit count")
        ax.legend(fontsize=7.5)
        ax.set_title(f"significant units, PLSC1 vs {lab_b}\n"
                     f"(both agents pooled)", fontsize=8.5)

    labels = ["original", "remove intra-net\ncoupling", "remove temporal\ncoupling"]
    vals = [dis["original"], dis["remove_intra"], dis["remove_temporal"]]
    axes[2].bar(range(3), vals, color=["#d62728", "0.5", "0.5"], width=0.6)
    axes[2].set_xticks(range(3)); axes[2].set_xticklabels(labels, fontsize=7.5)
    axes[2].set_ylabel("PLSC1 cross-covariance\n(observed $-$ shuffle)")
    axes[2].axhline(0, color="k", ls=":", lw=0.8)
    fig.tight_layout()
    return fig, dis


if plscs.get("social") is not None:
    fig, dis_soc = plot_weights_and_disruption("social")
    savefig(fig, "fig_weights_disruption.png")
    plt.show()

The two histograms separate cleanly: units that contribute significantly to PLSC1 sit at positive $|w_{PLSC1}| - |w_{U1}|$ (and $|w_{PLSC1}| - |w_{PLSC2}|$), the units significant for the comparison dimension sit at negative values, and almost no unit is shared between the populations — the shared and unique subspaces, and the first two shared dimensions, are carried by largely distinct sub-populations, as the paper found in mice and in agents (their Fig. 6i,j). And removing either the intra-network correlation structure or the temporal coupling between agents collapses the PLSC1 cross-covariance to shuffle level: the shared signal is a genuine, moment-by-moment coupling between the two networks, not an artifact of smooth dynamics.

### Partner representation in the full space, PLSC1, and the action space

How much of each network's activity encodes the *partner's* behavior? We regress the neural activity on behavior predictors — collision, self and partner positions, self and partner actions (one-hot), self and partner behaviors (new fields; approach for the chaser, escape for the explorer) — and measure the fraction of neural variance explained, with chance levels from circular time shifts. The **non-redundant partner contribution** is the drop in explained variance when the partner predictors are jointly permuted in time. We compute it in three neural spaces: the full hidden space, the top shared dimension PLSC1, and the **neural action space** $h_{action} = W_{out} W_{out}^\top h$ — the four-dimensional subspace that directly drives the policy.

In [ ]:
def get_partner_repr(condition, n_perm=100, seed=0):
    """Partner-behavior representation in full / PLSC1 / action space."""
    cache = ANALYSIS_DIR / f"partner_repr_{condition}.json"
    if cache.exists():
        return json.load(open(cache))
    if condition not in evals or plscs.get(condition) is None:
        return None
    ev, d = evals[condition], plscs[condition]
    out = {}
    for agent, tag in ((1, "explorer"), (2, "chaser")):
        R, keep = concat_activity(ev, agent=agent)
        pred = behavior_predictors(ev, agent=agent, keep=keep)
        w = d["W_X"] if agent == 1 else d["W_Y"]
        out[tag] = {
            "full_space": partner_representation(R, pred, n_perm, seed),
            "plsc1": partner_representation(_zscore(R) @ w[:, :1],
                                            pred, n_perm, seed),
            "action_space": partner_representation(
                neural_action_space(R, ev[f"W_out{agent}"]),
                pred, n_perm, seed),
        }
    json.dump(out, open(cache, "w"), indent=1)
    return out


partner_repr = {c: get_partner_repr(c) for c in evals}


def plot_partner_repr(partner_repr):
    fig, axes = plt.subplots(1, 3, figsize=(9.8, 2.8))
    spaces = [("full_space", "full neural space"),
              ("plsc1", "PLSC1"),
              ("action_space", "neural action space")]
    for ax, (space, title) in zip(axes, spaces):
        vals, labels, colors = [], [], []
        for cond, color in (("social", COLOR_SOCIAL),
                            ("nonsocial", COLOR_NONSOCIAL)):
            pr = partner_repr.get(cond)
            if pr is None:
                continue
            for tag in ("explorer", "chaser"):
                vals.append(100 * pr[tag][space]["partner_contribution"])
                labels.append(f"{cond}\n{tag}")
                colors.append(color)
        ax.bar(range(len(vals)), vals, color=colors, width=0.6)
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(labels, fontsize=6.5)
        ax.axhline(0, color="k", ls=":", lw=0.8)
        ax.set_title(title, fontsize=9)
        ax.set_ylabel("neural variance explained\nby partner (%)"
                      if ax is axes[0] else "")
    fig.tight_layout()
    return fig


if any(partner_repr.values()):
    fig = plot_partner_repr(partner_repr)
    savefig(fig, "fig_partner_representation.png")
    plt.show()

**Reading the result.** A few percent of the neural variance is explained by the partner's behavior *beyond* everything explained by the agent's own behavior — and this non-redundant partner representation is concentrated in the shared subspace (PLSC1) and, critically, survives projection into the action space: the partner representation is not a passive echo of the input but is positioned to drive behavior. In non-social agents it is at zero in every space. The paper's Fig. 6q adds a scatter across training checkpoints showing that the partner representation grows together with collision frequency; the next cell reproduces that view from our saved checkpoints.

In [ ]:
def partner_repr_over_training(condition, iters=(200, 1000, 2000, 4000, 6000),
                               n_episodes=10, n_perm=30, seed=0):
    """Action-space partner representation vs collision rate, across
    training checkpoints (Fig. 6p-r analog; needs checkpoint-iterK dirs)."""
    cache = ANALYSIS_DIR / f"partner_repr_traj_{condition}.json"
    if cache.exists():
        return json.load(open(cache))
    rows = []
    for it in iters:
        d0 = PAIR_DIRS[condition] / f"checkpoint-iter{it}"
        if not (d0 / "agent1").exists():
            continue
        a1 = AutoModel.from_pretrained(d0 / "agent1")
        a2 = AutoModel.from_pretrained(d0 / "agent2")
        ev = collect_rollouts(a1, a2, condition=condition,
                              n_episodes=n_episodes, episode_len=500,
                              seed=seed + it, device=DEVICE)
        R2, keep = concat_activity(ev, agent=2)
        if keep.sum() < 2 or R2.shape[0] < 200:
            # early checkpoints can produce degenerate (near-stationary)
            # policies; with all episodes excluded there is nothing to fit
            print(f"[{condition}] iter {it}: only {int(keep.sum())} usable "
                  f"episodes after degenerate-episode exclusion — skipped")
            continue
        pred = behavior_predictors(ev, agent=2, keep=keep)
        pr = partner_representation(
            neural_action_space(R2, ev["W_out2"]), pred, n_perm, seed)
        coll = float(ev["Bv2"][..., 2].sum(1).mean()) / 500 * 100
        rows.append({"iteration": it,
                     "action_space_partner": pr["partner_contribution"] * 100,
                     "collision_pct": coll})
        print(f"[{condition}] iter {it}: partner {rows[-1]['action_space_partner']:.2f}% "
              f"| collisions {coll:.1f}%")
    json.dump(rows, open(cache, "w"), indent=1)
    return rows


def plot_partner_traj(trajs):
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.8))
    for cond, color in (("social", COLOR_SOCIAL), ("nonsocial", COLOR_NONSOCIAL)):
        rows = trajs.get(cond)
        if not rows:
            continue
        it = [r["iteration"] for r in rows]
        pr = [r["action_space_partner"] for r in rows]
        axes[0].plot(it, pr, "o-", color=color, ms=3, label=cond)
        ax = axes[1] if cond == "social" else axes[2]
        coll = [r["collision_pct"] for r in rows]
        ax.scatter(coll, pr, color=color, s=12)
        if len(rows) > 2:
            b = np.polyfit(coll, pr, 1)
            xs = np.linspace(min(coll), max(coll), 10)
            ax.plot(xs, np.polyval(b, xs), color="k", lw=0.8)
            r2 = np.corrcoef(coll, pr)[0, 1] ** 2
            ax.set_title(f"{cond}: $R^2$ = {r2:.2f}", fontsize=8.5)
        ax.set_xlabel("collision time (%)", fontsize=8)
        ax.set_ylabel("partner repr. (%)", fontsize=8)
    axes[0].set_xlabel("training iteration")
    axes[0].set_ylabel("action-space partner repr. (%)")
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    return fig


trajs = {c: partner_repr_over_training(c) for c in evals}
if any(trajs.values()):
    fig = plot_partner_traj(trajs)
    savefig(fig, "fig_partner_traj.png")
    plt.show()

### The causal test: projecting out the shared subspace

Correlation is not causation — the shared subspace might be an epiphenomenon of two agents solving similar problems. The decisive experiment of the paper removes the shared signal *online*: before the chaser's action logits are computed, its hidden state is projected onto the null space of the top 10 shared dimensions,

$$h_t \leftarrow (I - P P^\top)\, h_t, \qquad P = \big[w_{Y,1}, \dots, w_{Y,10}\big],$$

while the recurrent dynamics themselves are left untouched. The control removes the same amount of variance in *non-overlapping* directions: 25 random principal components computed from temporally permuted, null-space-projected activity. If the shared subspace mediates social behavior, removing it should specifically impair chasing — fewer collisions, less partner-in-vision time, larger distances — while removing random directions should not.

In [ ]:
def run_perturbation(condition="social", n_episodes=100, episode_len=100,
                     seed=0):
    """Fig. 6s-u: rollouts with top-10 PLSC dims vs 25 random PCs projected
    out of the chaser (cached)."""
    outs = {}
    for tag in ("original", "plsc", "control"):
        cache = ANALYSIS_DIR / f"perturb_{condition}_{tag}.npz"
        if cache.exists():
            outs[tag] = load_rollouts(cache)
            continue
        if pairs.get(condition) is None or plscs.get(condition) is None:
            return None
        d = plscs[condition]
        R2, _ = concat_activity(evals[condition], agent=2)
        dirs = perturbation_directions(R2, d["W_Y"], n_plsc=10, n_control=25,
                                       seed=seed)
        P = {"original": None, "plsc": dirs["plsc"],
             "control": dirs["control"]}[tag]
        print(f"[{condition}] perturbation '{tag}' "
              f"(var removed: plsc {dirs['var_plsc']:.3f}, "
              f"control {dirs['var_control']:.3f}) ...")
        ev = collect_rollouts(*pairs[condition], condition=condition,
                              n_episodes=n_episodes, episode_len=episode_len,
                              seed=seed + 7, device=DEVICE, project_out2=P)
        save_rollouts(ev, cache)
        outs[tag] = ev
    return outs


def plot_perturbation(outs):
    metrics = [("collisions", "collisions / episode"),
               ("partner_in_vision", "time partner in vision"),
               ("mean_distance", "mean distance to partner")]
    colors = {"original": "0.4", "plsc": "#d62728", "control": "#1f77b4"}
    fig, axes = plt.subplots(1, 3, figsize=(8.6, 2.8))
    for ax, (key, label) in zip(axes, metrics):
        vals = [playoff_summary(outs[t])[key]
                for t in ("plsc", "control", "original")]
        bp = ax.boxplot(vals, patch_artist=True, widths=0.6,
                        medianprops=dict(color="k"))
        for patch, t in zip(bp["boxes"], ("plsc", "control", "original")):
            patch.set_facecolor(colors[t]); patch.set_alpha(0.75)
        ax.set_xticklabels(["PLSCs\nremoved", "random PCs\nremoved",
                            "original"], fontsize=8)
        ax.set_ylabel(label, fontsize=8)
    fig.tight_layout()
    return fig


perturbs = run_perturbation("social") if "social" in evals else None
if perturbs:
    fig = plot_perturbation(perturbs)
    savefig(fig, "fig_perturbation.png")
    plt.show()
    for tag, d in perturbs.items():
        s = playoff_summary(d)
        print(f"{tag:9s} collisions {s['collisions'].mean():5.2f} | "
              f"vision {s['partner_in_vision'].mean():.2f} | "
              f"distance {s['mean_distance'].mean():.2f}")

Removing the top ten shared dimensions from the chaser — directions that carry only a fraction of the total neural variance — sharply reduces collisions and partner-in-vision time and increases the inter-agent distance, while removing a *larger* amount of variance in random non-overlapping directions leaves behavior nearly intact. The shared subspace is therefore not a byproduct of social behavior but part of its mechanism: the chaser's policy reads the partner representation out of exactly these dimensions.

## Summary

We re-implemented the MARL half of Zhang et al. (2025) end-to-end in the NeuralRNN framework and reproduced its three layers of findings:

- **Behavior.** Independent PPO training of two ReLU RNNs in the chase arena yields role-specific social strategies: the chaser learns to find, track, and catch; the explorer learns to flee. Playoffs against random opponents and movement flow fields show these are learned social behaviors, absent under non-social training (Fig. 5 analogs).
- **Shared dynamics.** The two networks develop a large shared subspace — hundreds of PLSC dimensions whose activity correlates moment-by-moment across agents — which is essentially absent without social reward and partner vision, is carried by partially distinct units, and collapses when intra-network correlation or temporal coupling is disrupted (Fig. 6a–k analogs).
- **Causal role.** The partner's behavior is encoded non-redundantly in the shared subspace and in the action space, and projecting the top shared dimensions out of the chaser's hidden state before its policy head selectively dismantles chasing (Fig. 6l–u analogs).